# Antarctic lateral drag — supporting diagnostics

Companion to `LD-pub-workspace.ipynb`. This notebook owns the common-support FIP comparison, mechanics and thickness, actual lateral-drag face stresses, activation/retention, geometric localisation and pack checks. It uses the same configuration, experiments, periods and export directory. Run the main notebook first for the experiment manifest and FIA tables.

The old performance and paper2 notebooks mixed 1993 pilot runs, legacy APIs, estimated stress proxies, thickness exploration and wave forcing. Relevant publication diagnostics are retained below; pilot and out-of-scope wave recipes remain recoverable in Git history. Neither old results nor unverified case aliases are used as current evidence. All numerical stages default off until their input definitions are reviewed.

## 0. Shared configuration

In [2]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shlex
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display

# Fixed checkout locations, independent of Jupyter's working directory.
D_SRC     = Path.home() / "src"
D_shuga   = D_SRC / "shuga"
D_NB_ROOT = D_SRC / "CICE_free-slip" / "notebooks"

for required in [D_NB_ROOT / "ld_pub_workflow.py", D_shuga / "__init__.py"]:
    if not required.is_file():
        raise FileNotFoundError(f"Required file missing: {required}")

# Add paths BEFORE importing either local module.
sys.path.insert(0, str(D_SRC))
sys.path.insert(0, str(D_NB_ROOT))
import ld_pub_workflow as wf
import shuga
from shuga.core.types import RunSpec, ClassificationSpec, MetricsSpec
from shuga.core.paths import ShugaPaths
from shuga.core.regions import ANTARCTIC_8_REGIONS

print("Notebook directory:", D_NB_ROOT)
print("Publication helper:", wf.__file__)
print("Shuga package:", shuga.__file__)

Notebook directory: /home/581/da1339/src/CICE_free-slip/notebooks
Publication helper: /home/581/da1339/src/CICE_free-slip/notebooks/ld_pub_workflow.py
Shuga package: /home/581/da1339/src/shuga/__init__.py


In [3]:
CONFIG_PATH        = Path(os.environ.get('LD_PUB_CONFIG', D_NB_ROOT/'ld_pub_config.json'))
CFG                = json.loads(CONFIG_PATH.read_text())
START, END         = CFG['model_start'], CFG['model_end']
OBS_START, OBS_END = CFG['evaluation_start'], CFG['evaluation_end']
GROUP, METHOD      = CFG['experiment_group'], CFG['method']
BANK               = Path(CFG['afim_output_root'])
ARCHIVE            = BANK
EXPORT             = Path(os.environ.get('LD_PUB_EXPORT', str(Path(CFG['graphics_root'])/GROUP/'publication')))
EXPORT.mkdir(parents = True, exist_ok = True)
CACHE = EXPORT/'cache'
CACHE.mkdir(exist_ok = True)
manifest         = wf.expected_experiments()
EXPERIMENTS      = manifest.experiment.tolist()
FAMILIES         = {family: part.experiment.tolist() for family, part in manifest.groupby('family', sort=False)}
REFERENCE        = 'no-slip-LFI-match'
PRIMARY          = ['no-slip-LFI-match','free-slip-no-drag','Cs-high','Cq-high']
TENSILE_CASES    = ['Cs-high-ktens-zero','Cs-high-ktens-low','Cs-high-ktens-mid','Cs-high']
DIAGNOSTIC_CASES = list(dict.fromkeys(PRIMARY+TENSILE_CASES))
PAIRS            = {'boundary_only'            : ('free-slip-no-drag', 'no-slip-LFI-match'),
                    'static_drag'              : ('Cs-high', 'free-slip-no-drag'),
                    'quadratic_drag'           : ('Cq-high', 'free-slip-no-drag'),
                    'tensile_term'             : ('Cs-high', 'Cs-high-ktens-zero'),
                    'coast_increment_over_gib' : ('Cs-high', 'GIB-drag'),
                    'gib_increment_over_coast' : ('Cs-high', 'cst-drag')}
REGIONS          = list(ANTARCTIC_8_REGIONS)
OBS_FIA_PATH     = Path(CFG['obs_fia_path'])
UNIT_OVERRIDES   = {}  # Only documented metadata corrections; never inferred from values.
OPEN, COVERAGE, ARTIFACTS = {}, [], []

In [4]:
def contexts(name, method = METHOD, ice_type = 'FI'):
    run = RunSpec(name, START, END, project = CFG['project'], user = CFG['user'], experiment_group = GROUP)
    cls = ClassificationSpec(ice_type     = ice_type,
                             methods      = method,
                             grid_type    = CFG['grid_type'],
                             ispd_thresh  = CFG['ispd_thresh'],
                             aice_thresh  = CFG['aice_thresh'],
                             bin_window   = CFG['bin_window'],
                             bin_min_days = CFG['bin_min_days'],
                             roll_window  = CFG['roll_window'])
    met   = MetricsSpec(methods = method)
    paths = ShugaPaths(run_cfg          = run,
                       cls_cfg          = cls,
                       met_cfg          = met,
                       afim_output_root = BANK,
                       archive_root     = ARCHIVE,
                       graphics_root    = CFG['graphics_root'])
    return dict(run_cfg = run, cls_cfg = cls, met_cfg = met, pth_cfg = paths)

def load(name, kind, variables, method = METHOD, ice_type = 'FI'):
    key = (name, kind, tuple(variables), method, ice_type)
    if key not in OPEN:
        kwargs = contexts(name, method, ice_type)
        if kind!='history':
            kwargs['classification'] = method
        loader = {'metrics' : shuga.load_metrics,
                  'mask'    : shuga.load_classified,
                  'history' : shuga.load_cice}[kind]
        ds      = loader(**kwargs, variables = variables, dt0_str = START, dtN_str = END)
        missing = set(variables) - set(ds.variables)
        if missing:
            ds.close()
            raise KeyError(f'{name}: missing {missing}')
        OPEN[key] = ds
    return OPEN[key]

def open_product(path):
    path = Path(path)
    return xr.open_zarr(path, consolidated = False) if path.suffix == '.zarr' else xr.open_dataset(path)

def audited(da, name):
    da = da.copy(deep = False)
    if (name, da.name) in UNIT_OVERRIDES:
        da.attrs = dict(da.attrs, units = UNIT_OVERRIDES[(name, da.name)])
    return da

def git_commit(path):
    try:
        return subprocess.check_output(['git','-C', str(path), 'rev-parse', 'HEAD'], text = True, stderr = subprocess.DEVNULL).strip()
    except (OSError,subprocess.CalledProcessError):
        return 'UNVERIFIED'

def register(path, diagnostic, cases, period = None, notes = ''):
    period = period or (OBS_START, OBS_END)
    ARTIFACTS.append(dict(path        = str(path),
                          diagnostic  = diagnostic,
                          experiments = list(cases),
                          start       = period[0],
                          end         = period[1],
                          method      = METHOD,
                          signature   = SIGNATURE,
                          notes       = notes))

def export_table(table, stem, cases, period = None):
    period = period or (OBS_START, OBS_END)
    path   = EXPORT/f'{stem}_{period[0]}_{period[1]}.csv'
    table.to_csv(path, index = False)
    register(path, stem, cases, period)
    return path

In [5]:
provenance = dict(config                  = CFG,
                  helper_sha256           = wf.file_sha256(D_NB_ROOT / 'ld_pub_workflow.py'),
                  model_repository_commit = git_commit(D_NB_ROOT.parent),
                  shuga_module            = shuga.__file__,
                  shuga_commit            = git_commit(Path(shuga.__file__).parent),
                  unit_overrides          = {str(k):v for k,v in UNIT_OVERRIDES.items()})
SIGNATURE  = wf.signature(provenance)
print(f'Group: {GROUP}; model diagnostics: {START}–{END}; observational skill: {OBS_START}–{OBS_END}')
print('Inputs:',BANK/GROUP,'Outputs:',EXPORT)

Group: lateral-drag; model diagnostics: 2000-01-01–2005-12-31; observational skill: 2000-04-01–2005-04-30
Inputs: /g/data/gv90/da1339/afim_output/lateral-drag Outputs: /g/data/gv90/da1339/GRAPHICAL/lateral-drag/publication


## Diagnostic notebook settings

I'll use the following cell to select the diagnostics to run, supply their input files, and record the definitions I've checked. Run the shared configuration cell first: it defines the experiments, comparison pairs, dates, loaders and export directory.

For an initial run, leave all `RUN_*` switches `False`. Then configure and enable the stages needed for the current comparison. `None` means a required path or convention is unspecified; an empty `FIP_INPUTS` means no common-grid products have been selected.

The `*_VERIFIED` flags record checks I've completed. Setting a flag to `True` does not perform that validation.

### Selecting a diagnostic

| Switch | Purpose | Inputs and checks needed |
|---|---|---|
| `RUN_FIP` | Quantify spatial persistence errors and produce paired comparison panels. | `FIP_INPUTS` for all `DIAGNOSTIC_CASES`, `REGION_MASKS_PATH`, and verified common grid and temporal support. |
| `RUN_MECHANICS` | Compare FI thickness, deformation, strength and speed. | Native grid/area file, verified grid alignment, reviewed `HI_CONVENTION`, and the requested history fields. |
| `RUN_STRESS` | Summarise actual lateral-drag stresses. | Native grid/area file, verified grid alignment and stress definitions, and the four archived face-stress components. |
| `RUN_PACK` | Check surrounding-pack properties and total ice area/volume. | Native grid/area file, verified grid alignment, fixed pack-domain mask and reviewed thickness convention. |
| `RUN_OCCURRENCE` | Calculate annual FI-day counts and first/last occurrence indices. | Complete daily binary FI masks. These indices are not automatically validated formation or retreat dates. |
| `RUN_ANCHOR` | Relate persistence to proximity to mechanical anchors. | Native grid/area file and verified production form-factor file. |
| `RUN_POINT_OBS` | Compare thickness at selected observation locations and dates. | Reviewed observation CSV, thickness convention, and native grid file containing an ocean mask. |
| `RUN_TENDENCIES` | Compare thermodynamic and dynamic area tendencies within FI. | Native cell areas, known tendency units, and verified mask/tendency temporal support. |
| `RUN_PREPARE_FIP` | Prepare daily common-grid inputs before calculating persistence. | Reviewed daily observation product, native grid file, and verified grid/support and remapping choices. |

### Definitions and input products

I'll set `HI_CONVENTION` from the executed history-field definition:

- `'ice-covered-area'`: use `hi` directly.
- `'grid-cell-volume-per-area'`: derive ice-covered-area thickness as `hi / aice`, where concentration is positive.

Native-grid files serve model diagnostics; common-grid files serve observational FIP comparisons. Matching array dimensions alone does not establish physical alignment.

For FIP, I'll either supply reviewed products through `FIP_INPUTS` or enable `RUN_PREPARE_FIP` to build them. Preparation populates `FIP_INPUTS`; enable `RUN_FIP` as well to calculate skill. The observation product must document its source, projection and temporal-support policy. I'll establish how the original composites represent time before supplying a daily product.

`FIP_PIXEL_SIZE_M` must match the observation grid. I'll justify the nearest-neighbour search radius and retain missing values where no suitable source cell exists. I'll leave the longitude shift at zero unless the actual coordinates justify a correction. A finer comparison grid does not increase the model's resolution.

### Regions, comparisons and reuse

`DIAGNOSTIC_REGION` selects the mechanics/stress crop; FIP scores and maps iterate over `REGIONS`. `MAP_PAIRS` selects FIP panels and mechanics differences. Each tuple is `(target, reference)`, so differences are target minus reference. Changing these pairs does not automatically add experiments to `DIAGNOSTIC_CASES`.

I'll keep `RECOMPUTE_MECHANICS=True` after changing source data or definitions. Setting it `False` permits reuse of a matching cache, but directory timestamps cannot detect every rewritten Zarr chunk. The exact-zero-speed summary is generated during recomputation.

After changing these settings, I'll restart and run from the top to avoid mixing old datasets with new settings. Expensive remapping and reductions belong in an allocated analysis/PBS job. These switches do not submit jobs or alter model inputs.

In [6]:
# Analysis stages: enable the diagnostics needed for the current comparison.
RUN_FIP                       = False  # Compute common-support FIP skill and paired difference panels.
RUN_MECHANICS                 = False  # Reduce FI thickness, deformation, strength and T-grid speed.
RUN_STRESS                    = False  # Summarise actual lateral-drag stresses at E/N velocity faces.
RUN_PACK                      = False  # Compare fixed-domain/pack properties and total SIA/SIV.
RUN_OCCURRENCE                = False  # Export annual FI-day counts and first/last occurrence indices.
RUN_ANCHOR                    = False  # Relate FI persistence to distance from anchor-bearing cells.
RUN_POINT_OBS                 = False  # Sample thickness at reviewed observation sites and dates.
RUN_TENDENCIES                = False  # Sum FI-conditioned area tendencies; not a closed FIA budget.
RECOMPUTE_MECHANICS           = True   # Rebuild reductions; False permits matching-cache reuse.

# Review flags: True records a completed scientific check.
FIP_GRID_AND_SUPPORT_VERIFIED  = False  # Confirm common grid, dates, valid-day support and binary masks.
HI_CONVENTION                  = None   # 'ice-covered-area' or 'grid-cell-volume-per-area'.
LD_STRESS_DEFINITION_VERIFIED  = False  # Confirm KuxE/KuyE/KuxN/KuyN are actual stresses in Pa.
NATIVE_GRID_VERIFIED           = False  # Confirm native coordinates, areas and history/mask alignment.
POINT_OBS_SAMPLING_VERIFIED    = False  # Review observation timing, location, thickness/snow and sampling.
FORM_FACTOR_GRID_VERIFIED      = False  # Confirm production FFx/FFy identity and native-grid alignment.
MASK_TEMPORAL_SUPPORT_VERIFIED = False  # Confirm tendency/mask timestamps and averaging intervals agree.

# Input paths: supply the products needed by the enabled stages.
NATIVE_AREA_PATH               = None   # Native NetCDF: tarea [m²], TLON, TLAT; ocean_mask for point matching.
REGION_MASKS_PATH              = None   # Common-grid binary sector masks named by REGIONS; review overlap/gaps.
FIXED_PACK_MASK_PATH           = None   # Native fixed_pack_mask, chosen independently of experiment outcomes.
FIP_INPUTS                     = {}     # Experiment -> daily common-grid mod_mask, obs_mask and area [m²].
FORM_FACTOR_PATH               = None   # Executed production NetCDF containing FFx and FFy.
POINT_OBS_PATH                 = None   # CSV: site,time,lon,lat,thickness_m; optional snow_m for context.
STRENGTH_VARIABLE              = "strength"  # History strength variable; explicit units are required.
DIAGNOSTIC_REGION              = "Aus"   # Region key for mechanics/stress crops; repeat other sectors as needed.

# Comparison tuples are (target, reference); differences are target minus reference.
MAP_PAIRS = {"boundary_only"  : PAIRS["boundary_only"],   # Free-slip minus matched no-slip; both without drag.
             "static_drag"    : PAIRS["static_drag"],     # Cs-high minus free-slip-no-drag.
             "quadratic_drag" : PAIRS["quadratic_drag"],  # Cq-high minus free-slip-no-drag.
             "tensile_term"   : PAIRS["tensile_term"]}    # Cs-high minus Cs-high-ktens-zero.

# Optional preparation: remap daily masks before calculating persistence.
RUN_PREPARE_FIP               = False  # Build common-grid inputs in monthly batches; populate FIP_INPUTS.
OBS_DAILY_COMMON_PATH          = None   # EPSG:3031 obs_mask(time,y,x), area [m²], x/y [m], source/support attrs.
FIP_REMAP_VERIFIED            = False  # Confirm projection, coordinates, remapping and missing-data treatment.
FIP_PIXEL_SIZE_M              = 5000   # Target projected spacing [m]; must match the observation grid.
FIP_RADIUS_M                  = 10000  # Maximum nearest-neighbour search distance [m].
CICE_LONGITUDE_SHIFT_DEG       = 0.0    # Additive longitude correction [degrees]; requires coordinate evidence.

## 1. FIP differences and regional maps

### Checks before generating FIP differences

Before enabling generation, I'll check:

- **Dates and coverage:** the requested period is available in both sources, with missing samples and AF2020 composite timing understood. The workflow averages each source over its own available samples; it does not enforce shared daily denominators.
- **Classification:** the model masks use the intended method, grid, speed threshold and persistence-filter settings.
- **Physical alignment:** the native model coordinates and AF2020 comparison grid represent the intended locations. I'll inspect coastal alignment and justify any longitude correction, including the PBS workflow's default +0.25° shift.
- **Remapping:** the target pixel spacing matches the observation product, and the nearest-neighbour search radius is appropriate. Unmatched pixels should remain missing.
- **Units and validity:** model and observed persistence are fractions from 0 to 1. I'll distinguish valid zero persistence from missing observations and check the interpretation of the source FIC values.
- **Existing outputs:** any reused comparison store has the intended inputs and settings. Metadata checks alone cannot detect changes to source data; I'll regenerate affected products when necessary.

I'll record any decisions affecting interpretation alongside the resulting tables and figures. These checks apply to the existing shuga persistence workflow; they do not require a reconstructed daily observation product or a `common_grid_verified` attribute.

Once these checks are complete, I'll run the following cells to generate or reuse the numerical products, inspect the regional statistics, and produce the PyGMT maps.

I'll use the existing shuga workflow in `scripts/comparisons/FIP_differencing.py` and the PyGMT figure generator in `CICEPlotter.plot_fip`. This keeps the calculation, regional projections, coastlines, colour palettes and colour bars consistent with the established PBS workflow.

For the selected comparison period, shuga calculates model persistence from the classified FI masks as a fraction, and observed persistence from the mean AF2020 common-grid `FIC`. It then remaps the model persistence field onto the observation grid using nearest-neighbour sampling and calculates **model minus AF2020**. Both persistence fields remain in the range 0–1; positive differences indicate greater model persistence.

I'll check the model and observation coverage, missing-data treatment and AF2020 composite timing before interpreting the results. The two sources retain their respective temporal sampling: matching the date limits does not establish identical valid-day denominators. These cells follow the established persistence comparison without reconstructing daily observations.

### Running the three cells

1. **Configure and generate:** set `RUN_PREPARE_FIP=True` to generate missing shuga comparison products. Existing stores are reused unless `FIP_OVERWRITE=True`. I'll review the inputs and settings before deliberately regenerating a product.
2. **Load and summarise:** set `RUN_FIP=True` to load the comparison fields, check their recorded settings and calculate shuga's persistence-weighted regional overlap and excess statistics.
3. **Plot:** generate the regional maps through `CICEPlotter.plot_fip`, using PyGMT throughout. Set `FIP_SHOW_MAPS=True` to display the saved maps in the notebook.

I'll verify the target-grid spacing, projection, nearest-neighbour search radius and model longitude coordinates. The PBS workflow defaults to a +0.25° longitude shift; the notebook passes `CICE_LONGITUDE_SHIFT_DEG` explicitly. I'll use the value justified by the actual coordinates and record it with the outputs.

### Products and interpretation

The numerical products contain `mod`, `obs`, `diff` and `diff_cat`, with geographic coordinates and comparison metadata. They are written beneath each experiment's `zarr/comparisons` directory. Regional maps are written beneath its `GRAPHICAL` figure directory in `FIP_diff/<region>/`.

The continuous maps show persistence differences. Shuga masks cells where both sources have zero persistence in `diff`, while retaining their values in `mod` and `obs`. The categorical field uses the declared difference threshold to distinguish agreement, model-dominant and observation-dominant persistence.

The regional statistics follow shuga's existing persistence-weighted overlap/excess method and configured comparison-pixel area. I'll check the area approximation and regional bounds before quoting quantitative areas. Its `ANT` row sums the regional results, so I'll check regional overlap and coverage before describing it as a circum-Antarctic total.

These replacement cells do not use `FIP_INPUTS`, `OBS_DAILY_COMMON_PATH` or `REGION_MASKS_PATH`. The earlier daily-mask preparation is superseded for this section. Expensive generation belongs in an allocated analysis/PBS job; existing numerical products can be loaded and replotted without repeating the remapping.

In [7]:
# Use the existing standalone shuga comparison script.
FIP_SCRIPT             = (Path(D_shuga).expanduser() / "scripts" / "comparisons" / "FIP_differencing.py")
AF2020_FIC_STORE       = Path("/g/data/gv90/da1339/SeaIce/FI_obs/AF-FI-2020db_common-5km_pyresample.zarr")
FIP_OVERWRITE          = False  # Explicitly allow regeneration of existing derived products.
FIP_CATEGORY_THRESHOLD = 0.5  # Shuga difference-category threshold; not an occurrence threshold.
FIP_SHOW_MAPS          = False  # Display the saved PyGMT PNGs inside the notebook.
FIP_CASES              = list(dict.fromkeys(DIAGNOSTIC_CASES))
FIP_PRODUCTS           = {}

# Ensure child Python processes can import this standalone shuga checkout.
fip_env        = os.environ.copy()
fip_pythonpath = [str(Path(D_shuga).expanduser().resolve().parent), fip_env.get("PYTHONPATH", "")]
fip_env["PYTHONPATH"] = os.pathsep.join(value for value in fip_pythonpath if value)

for name in FIP_CASES:
    ctx   = contexts(name)
    paths = ctx["pth_cfg"]
    # Same numerical location and filename convention as FIP_differencing.py.
    out_store = (paths.zarr_root / "comparisons" / 
                 (f"FIPdiff_{name}_minus_AF2020_{METHOD}_{CFG['grid_type']}_{OBS_START}_{OBS_END}.zarr"))
    FIP_PRODUCTS[name] = {"store"     : out_store,
                          "stats"     : out_store.with_suffix(".regional_stats.csv"),
                          "plot_root" : paths.figure_root() / "FIP_diff"}

if RUN_PREPARE_FIP:
    if not FIP_SCRIPT.is_file():
        raise FileNotFoundError(FIP_SCRIPT)
    if not AF2020_FIC_STORE.is_dir():
        raise FileNotFoundError(AF2020_FIC_STORE)
    for name, product in FIP_PRODUCTS.items():
        if product["store"].exists() and not FIP_OVERWRITE:
            print(f"Using existing product: {product['store']}")
            continue
        command = [sys.executable, str(FIP_SCRIPT),
                   "--sim-name"             , name,
                   "--experiment-group"     , GROUP,
                   "--start-date"           , START,
                   "--end-date"             , END,
                   "--fip-start"            , OBS_START,
                   "--fip-end"              , OBS_END,
                   "--classification"       , METHOD,
                   "--grid-type"            , CFG["grid_type"],
                   "--ice-type"             , "FI",
                   "--hemisphere"           , "SH",
                   "--project"              , CFG["project"],
                   "--user"                 , CFG["user"],
                   "--afim-output-root"     , str(BANK),
                   "--af2020-store"         , str(AF2020_FIC_STORE),
                   "--ispd-thresh"          , str(CFG["ispd_thresh"]),
                   "--bin-window"           , str(CFG["bin_window"]),
                   "--bin-min-days"         , str(CFG["bin_min_days"]),
                   "--roll-window"          , str(CFG["roll_window"]),
                   "--pixel-size-m"         , str(FIP_PIXEL_SIZE_M),
                   "--radius-of-influence-m", str(FIP_RADIUS_M),
                   "--cice-lon-shift-deg"   , str(CICE_LONGITUDE_SHIFT_DEG),
                   "--category-threshold"   , str(FIP_CATEGORY_THRESHOLD),
                   "--out-store"            , str(product["store"]),
                   "--stats-csv"            , str(product["stats"])]

        if FIP_OVERWRITE:
            command.append("--overwrite")

        # Numerics here; the third cell calls the existing PyGMT plotter.
        print(f"\nGenerating shuga FIP differences: {name}", flush=True)
        subprocess.run(command, cwd = FIP_SCRIPT.parent, env = fip_env, check = True)
else:
    print("FIP generation disabled. Set RUN_PREPARE_FIP=True to generate "
          "missing products; RUN_FIP=True can load existing products.")

FIP generation disabled. Set RUN_PREPARE_FIP=True to generate missing products; RUN_FIP=True can load existing products.


In [8]:
from shuga.regridding.pyresample import compute_fipdiff_stats_weighted

FIP_DATA  = {}
FIP_STATS = None

if RUN_FIP:
    tables = []
    for name, product in FIP_PRODUCTS.items():
        path = product["store"]
        if not path.is_dir():
            raise FileNotFoundError(f"{name}: missing FIP comparison product:\n{path}\n"
                                    "Enable RUN_PREPARE_FIP and run the preceding cell first.")
        with xr.open_zarr(path, consolidated=False) as source:
            required = {"mod", "obs", "diff", "diff_cat", "lon", "lat"}
            missing  = required - set(source.variables)
            if missing:
                raise KeyError(f"{path}: missing variables {sorted(missing)}")
            # Reject reuse of a product with different recorded settings.
            expected_text = {"sim_name"       : name,
                             "classification" : METHOD,
                             "grid_type"      : CFG["grid_type"],
                             "fip_start"      : OBS_START,
                             "fip_end"        : OBS_END,
                             "af2020_store"   : str(AF2020_FIC_STORE)}
            expected_numeric = {"pixel_size_m"          : FIP_PIXEL_SIZE_M,
                                "radius_of_influence_m" : FIP_RADIUS_M,
                                "cice_lon_shift_deg"    : CICE_LONGITUDE_SHIFT_DEG}
            for key, expected in expected_text.items():
                if str(source.attrs.get(key)) != str(expected):
                    raise ValueError(f"{name}: {key} differs from the current settings. "
                                     "Review the existing product before regenerating.")
            for key, expected in expected_numeric.items():
                actual = source.attrs.get(key)
                if actual is None or not np.isclose(float(actual), float(expected)):
                    raise ValueError(f"{name}: {key} differs from the current settings.")
            if not np.isclose(float(source.diff_cat.attrs.get("category_threshold", np.nan)), FIP_CATEGORY_THRESHOLD):
                raise ValueError(f"{name}: difference-category threshold changed")
            # These are reduced 2-D fields, not daily histories.
            dataset = source[["mod", "obs", "diff", "diff_cat"]].load()
        for variable in ("mod", "obs"):
            values = dataset[variable]
            if not bool(np.isfinite(values).any()):
                raise ValueError(f"{name}: {variable} has no finite persistence values")
            if bool(((values < 0) | (values > 1)).any()):
                raise ValueError(f"{name}: {variable} must be a 0–1 fraction, not percent")
        FIP_DATA[name] = dataset
        # Same regional statistics helper and options as the shuga script.
        table = compute_fipdiff_stats_weighted(dataset, pixel_size_m = float(FIP_PIXEL_SIZE_M), threecat_percent_only = False)
        table.to_csv(product["stats"])
        table = table.rename_axis("region").reset_index()
        table.insert(0, "experiment", name)
        tables.append(table)
        register(path, "shuga FIP differences", [name], (OBS_START, OBS_END),"Standard shuga workflow; fractions; model minus AF2020")
        register(product["stats"], "shuga persistence-weighted regional statistics", [name],(OBS_START, OBS_END))
    FIP_STATS = pd.concat(tables, ignore_index = True)
    display(FIP_STATS)
else:
    print("FIP loading and plotting disabled: RUN_FIP=False")

FIP loading and plotting disabled: RUN_FIP=False


In [9]:
from shuga.plotting.cice import CICEPlotter
from IPython.display import Image, display

FIP_MAPS = {}

if RUN_FIP and FIP_DATA:
    for name, dataset in FIP_DATA.items():
        # Reuse the notebook's experiment-group-aware configuration.
        plotter = CICEPlotter(**contexts(name))
        for region_name in REGIONS:
            region_bounds = ANTARCTIC_8_REGIONS[region_name]["plot_region"]
            output_path   = (FIP_PRODUCTS[name]["plot_root"] / region_name / f"{OBS_START}_{OBS_END}_{name}_{region_name}.png")
            output_path.parent.mkdir(parents=True, exist_ok=True)
            saved = plotter.plot_fip(method=METHOD,
                                     source="dataset",
                                     field="diff",
                                     dataset=dataset,
                                     regions={region_name: region_bounds},
                                     output_path=output_path,
                                     title=f"{name} - AF2020 {OBS_START} to {OBS_END}")
            FIP_MAPS[(name, region_name)] = Path(saved)
            register(saved, "shuga PyGMT FIP difference map", [name], (OBS_START, OBS_END), 
                     f"{region_name}; model minus AF2020; persistence fraction")
            print(f"Saved: {saved}")
            if FIP_SHOW_MAPS:
                display(Image(filename=str(saved)))

## 2. Mechanics, thickness and strength

I'll use shuga's existing temporal-mean and persistence calculations for FI-conditioned thickness and strength, its Tc velocity reconstruction, and its deformation helper. Figures will use `CICEPlotter.plot_triptych`, with PyGMT throughout.

The reductions use the same underlying methods as shuga's FIHI and FIST metrics. I'll calculate them from the selected daily histories and masks here so that the period, valid samples and reviewed thickness convention are explicit. Existing metric stores are interchangeable only after those definitions and their provenance have been checked.

FI-conditioned means describe conditions during valid FI samples. Persistence-weighted fields multiply that conditional mean by the FI fraction over the same field-valid samples. A valid cell with no FI receives zero weighted contribution; a cell with no valid samples remains missing. Thickness and strength can have different valid-sample support, which is retained separately.

I'll interpret hi according to the executed history definition. Strength retains its native units; vertically integrated strength in N/m is not MPa. Shuga's FIST spatial field and FIS strength time series are different quantities.

For deformation, I'll pass unit-normalised divu and shear to shuga's strain helper, deliberately excluding its sigP and shear-only fallbacks. The plotted statistic is the temporal Q90 of the daily-history proxy hypot(divu, shear), restricted to ice-covered samples. It is neither an FI-conditioned statistic nor a subdaily extreme.

Tc reconstruction is performed lazily on the full periodic grid before cropping. Because shuga's reconstruction replaces missing velocity components with zero, I'll also reconstruct a source-validity indicator and exclude samples with missing contributing components from the speed mean and zero-speed audit. Exactly zero reconstructed speed still does not, by itself, establish stationary or mechanically coherent ice.

I'll compare both matched controls, the active static/quadratic closures and the static tensile sequence. All matched differences are target minus reference. Figures share colour scales across experiments; difference figures use symmetric scales shared across comparisons.

The regional cache records the settings, shuga implementation hashes, source metadata and zero-speed audit. It does not hash every source-data chunk. I'll keep recomputation enabled after source changes, and rerun both cells after changing definitions or comparison settings.

In [10]:
import shuga.metrics.calculations as mcalc
import shuga.metrics.diagnostics as mdiag
import shuga.regridding.cice as cgrid

# Only unit conversion and publication-specific orchestration live here.
def mechanics_si(field, factors, output_units):
    unit = (str(field.attrs.get("units", "")).lower().replace(" ", ""))
    if unit not in factors:
        raise ValueError(f"{field.name}: unsupported or missing units {field.attrs.get('units')!r}")
    out = field * factors[unit]
    out.attrs = dict(field.attrs, units=output_units)
    return out

def mechanics_fi_maps(field, mask, conditional_name):
    """Use shuga reductions with identical field-valid denominators."""
    field, mask = xr.align(field, mask, join="exact")
    valid       = np.isfinite(field) & mask.notnull()
    conditional = mcalc.compute_temporal_mean(field.where(valid & (mask == 1)), name = conditional_name,
                                              long_name = f"{conditional_name}: mean during valid FI samples")
    persistence = mcalc.compute_persistence_mask(mask.where(valid), name = f"{conditional_name}_persistence",
                                                 long_name = f"FI fraction on valid {conditional_name} support",
                                                 percent = False)
    # No FI on valid support -> zero; no valid support -> missing.
    weighted = xr.where(persistence == 0, 0.0, conditional * persistence)
    weighted.attrs = dict(field.attrs, long_name = f"Persistence-weighted {conditional_name}")
    return conditional, weighted, persistence

In [11]:
MECHANICS = {}
zero_rows = []

if RUN_MECHANICS:
    if (HI_CONVENTION is None or NATIVE_AREA_PATH is None or not NATIVE_GRID_VERIFIED):
        raise ValueError("Review the hi convention, native grid and coordinates first")
    with open_product(NATIVE_AREA_PATH) as source:
        native = source[["tarea", "TLON", "TLAT"]].load()
    wf.area_m2(native.tarea)
    region = DIAGNOSTIC_REGION
    bounds = ANTARCTIC_8_REGIONS[region]["plot_region"]
    tile   = wf.crop_region(native, native.TLON, native.TLAT, bounds)
    velocity_names = ["uvelE", "uvelN", "vvelE", "vvelN"]
    velocity_factors = {"m/s": 1.0, "ms-1": 1.0, "ms^-1": 1.0, "cm/s": 0.01, "cms-1": 0.01, "cms^-1": 0.01}
    strain_factors = {"s-1": 1.0, "s^-1": 1.0, "s**-1": 1.0, "1/s": 1.0, "1/day": 1 / 86400, "day-1": 1 / 86400,
                      "d-1": 1 / 86400, "%/day": 0.01 / 86400, "%day-1": 0.01 / 86400, "%d-1": 0.01 / 86400}
    implementation = {module.__name__: wf.file_sha256(module.__file__) for module in (mcalc, mdiag, cgrid)}
    for name in DIAGNOSTIC_CASES:
        variables  = list(dict.fromkeys(["hi", "aice", "divu", "shear", STRENGTH_VARIABLE] + velocity_names))
        history    = load(name, "history", variables)
        classified = load(name, "mask", ["FI_mask"])
        history, classified = wf.align_daily(history, classified, start = START, end = END)
        h = wf.crop_region(history, native.TLON, native.TLAT, bounds)
        mask = wf.binary_mask(wf.crop_region(classified.FI_mask, native.TLON, native.TLAT, bounds))
        h, mask = xr.align(h, mask, join="exact")
        model_store = Path(contexts(name)["pth_cfg"].resolve_cice_store())
        sig = wf.signature({"workflow"          : "shuga-mechanics-v3",
                            "analysis"          : SIGNATURE,
                            "implementation"    : implementation,
                            "experiment"        : name,
                            "region"            : region,
                            "bounds"            : bounds,
                            "period"            : [START, END],
                            "hi_convention"     : HI_CONVENTION,
                            "strength_variable" : STRENGTH_VARIABLE,
                            "aice_threshold"    : CFG["aice_thresh"],
                            "native_grid"       : str(NATIVE_AREA_PATH),
                            "native_grid_mtime" : Path(NATIVE_AREA_PATH).stat().st_mtime_ns,
                            "history_attrs"     : dict(history.attrs),
                            "mask_attrs"        : dict(classified.attrs),
                            "variable_attrs"    : {variable: dict(history[variable].attrs) for variable in variables},
                            "model_store"       : str(model_store),
                            "model_store_mtime" : model_store.stat().st_mtime_ns})
        path = CACHE / (f"{name}_{region}_mechanics_{START}_{END}.nc")
        if path.exists() and not RECOMPUTE_MECHANICS:
            with wf.read_cached(path, sig) as cached:
                fields = cached.load()
            if "zero_speed_audit" not in fields.attrs:
                raise ValueError(f"{name}: cache lacks the speed audit; recompute")
            zero_rows.append(json.loads(fields.attrs["zero_speed_audit"]))
        else:
            hi = wf.ice_thickness(h, HI_CONVENTION)
            strength = h[STRENGTH_VARIABLE]
            if not strength.attrs.get("units"):
                raise ValueError(f"{name}: strength units must be explicit")
            hi_c, hi_w, hi_p = mechanics_fi_maps(hi, mask, "FIHI")
            st_c, st_w, st_p = mechanics_fi_maps(strength, mask, "FIST")
            # Force the divu/shear branch; do not substitute sigP.
            deformation = xr.Dataset({variable: mechanics_si(h[variable], strain_factors, "s-1") for variable in ("divu", "shear")})
            strain = mdiag.strain_invariant(deformation)
            strain = strain.where((h.aice > CFG["aice_thresh"]) & np.isfinite(strain))
            if strain.chunks is not None:
                strain = strain.chunk({"time": -1})
            strain_q90 = (strain.quantile(0.90, dim="time", skipna=True).drop_vars("quantile", errors="ignore") * 1e6)
            strain_q90.attrs = {"units": "1e-6 s^-1", "long_name": "Temporal Q90 of daily ice-covered strain proxy"}
            # Reconstruct on the full grid, then crop the lazy result.
            velocity = xr.Dataset({variable: mechanics_si(history[variable], velocity_factors, "m/s") for variable in velocity_names})
            speed_full = cgrid.compute_tgrid_speed(velocity, history.aice, grid_type = "Tc", wrap_x = True, cgrid_combine = "mean")
            # Apply the same stencil to 0/1 source-validity indicators.
            indicators = [np.isfinite(history[variable]).astype("float32") for variable in velocity_names]
            valid_e, valid_n = cgrid.c2t(*indicators, history.aice, wrap_x = True, combine = "mean", nan_to_zero = True)
            support_full = (valid_e == 1) & (valid_n == 1)
            speed = wf.crop_region(speed_full.where(support_full), native.TLON, native.TLAT, bounds)
            speed.attrs["units"] = "m/s"
            finite_ice = ((h.aice > CFG["aice_thresh"]) & np.isfinite(speed))
            zero = finite_ice & (speed == 0)
            counts = xr.Dataset({"valid_icy_samples": finite_ice.sum(),
                                 "exact_zero_samples": zero.sum(),
                                 "zero_samples_classified_FI": (zero & (mask == 1)).sum()}).compute()
            audit = {"experiment": name, "region": region, **{key: int(value.item()) for key, value in counts.data_vars.items()}}
            zero_rows.append(audit)
            speed_mean = mcalc.compute_temporal_mean(speed.where(finite_ice), name = "speed_mean",
                                                     long_name = "Mean Tc speed on valid ice-covered samples")
            fields = xr.Dataset({"hi_conditional"       : hi_c,
                                 "hi_weighted"          : hi_w,
                                 "fip"                  : hi_p,
                                 "strength_conditional" : st_c,
                                 "strength_weighted"    : st_w,
                                 "strength_support_fip" : st_p,
                                 "strain_q90"           : strain_q90,
                                 "speed_mean"           : speed_mean})
            fields = fields.assign_coords(TLON = tile.TLON, TLAT = tile.TLAT)
            fields.attrs["zero_speed_audit"] = json.dumps(audit)
            fields = fields.compute()
            wf.save_cached(fields, path, sig)
        MECHANICS[(name, region)] = fields
        register(path, "shuga mechanics reductions", [name], (START, END), region)
    export_table(pd.DataFrame(zero_rows), "exact_zero_speed_" + region, DIAGNOSTIC_CASES, (START, END))
else:
    print("Mechanics disabled: RUN_MECHANICS=False")

Mechanics disabled: RUN_MECHANICS=False


In [12]:
from shuga.plotting.cice import CICEPlotter

MECHANICS_DIFFERENCES = {}

if MECHANICS:
    region = DIAGNOSTIC_REGION
    bounds = ANTARCTIC_8_REGIONS[region]["plot_region"]
    panel_variables = ["strain_q90", "hi_weighted", "strength_weighted"]
    panel_labels = {"strain_q90"        : "Q90 strain proxy",
                    "hi_weighted"       : "Persistence-weighted thickness",
                    "strength_weighted" : "Persistence-weighted strength"}
    panel_cmaps = {"strain_q90"        : "cmocean/thermal",
                   "hi_weighted"       : "cmocean/matter",
                   "strength_weighted" : "cmocean/turbid"}
    # Construct matched contrasts, retaining units and physical coordinates.
    for label, (target, reference) in MAP_PAIRS.items():
        target_key    = (target, region)
        reference_key = (reference, region)
        if target_key not in MECHANICS or reference_key not in MECHANICS:
            print(f"Pending {label}: {target} / {reference}")
            continue
        a, b = xr.align(MECHANICS[target_key], MECHANICS[reference_key], join = "exact")
        for coordinate in ("TLON", "TLAT"):
            if not np.array_equal(a[coordinate].values, b[coordinate].values, equal_nan = True):
                raise ValueError(f"{label}: physical grid differs for {coordinate}")
        for variable in a.data_vars:
            if a[variable].attrs.get("units") != b[variable].attrs.get("units"):
                raise ValueError(f"{label}: incompatible units for {variable}")
        difference = a - b
        for variable in difference.data_vars:
            difference[variable].attrs = dict(a[variable].attrs)
            difference[variable].attrs["long_name"] = (f"{target} minus {reference}: {variable}")
        difference.attrs = {"target"               : target,
                            "reference"            : reference,
                            "difference_convention": "target minus reference",
                            "start"                : START,
                            "end"                  : END}
        path = EXPORT / (f"{label}_{region}_mechanics_difference_{START}_{END}.nc")
        wf.save_cached(difference, path, SIGNATURE)
        register(path, "target minus reference mechanics", [target, reference], (START, END), region)
        MECHANICS_DIFFERENCES[label] = difference

    def mechanics_scales(datasets, symmetric=False):
        """Common plotting limits; no new scientific reduction."""
        result = {}
        for variable in panel_variables:
            units = {ds[variable].attrs.get("units") for ds in datasets}
            if len(units) != 1 or None in units:
                raise ValueError(f"{variable}: missing or inconsistent plotting units")
            finite_parts = [ds[variable].values[np.isfinite(ds[variable].values)] for ds in datasets]
            finite_parts = [part for part in finite_parts if part.size]
            if not finite_parts:
                raise ValueError(f"{variable}: no finite values to plot")
            values = np.concatenate(finite_parts)
            if symmetric:
                limit = float(np.max(np.abs(values))) or 1.0
                lower, upper = -limit, limit
            else:
                lower = min(0.0, float(values.min()))
                upper = max(0.0, float(values.max()))
                if lower == upper:
                    upper = lower + 1.0
            result[variable] = [lower, upper, (upper - lower) / 20]
        return result

    def plot_mechanics_triptych(dataset, plotter, heading, path, scales, difference = False):
        panels = []
        for variable in panel_variables:
            units = dataset[variable].attrs["units"]
            label = panel_labels[variable]
            if difference:
                label = "Difference in " + label.lower()
            panels.append({"title": f"{heading}: {panel_labels[variable]}",
                           "layers": [{"data": dataset[variable],
                                       "lon": dataset.TLON,
                                       "lat": dataset.TLAT,
                                       "mask_zero": False,
                                       "cmap": ("polar" if difference else panel_cmaps[variable]),
                                       "series": scales[variable],
                                       "colorbar": {"position": "JBC+w6c/0.3c+h+o0c/0.8c",
                                                    "frame": [f"xaf+l{label} ({units})"]}}]})
        return plotter.plot_triptych(region      = bounds,
                                     panels      = panels,
                                     fig_size    = 10.0,
                                     panel_gap   = "2.5c",
                                     output_path = path)
    regional_data = [ds for (name, reg), ds in MECHANICS.items() if reg == region]
    case_scales = mechanics_scales(regional_data)
    for (name, reg), dataset in MECHANICS.items():
        if reg != region:
            continue
        plotter = CICEPlotter(**contexts(name))
        path = EXPORT / (f"{name}_{region}_mechanics_{START}_{END}.png")
        saved = plot_mechanics_triptych(dataset, plotter, name, path, case_scales)
        register(saved, "shuga PyGMT mechanics triptych", [name], (START, END), region)
        print(f"Saved: {saved}")
    if MECHANICS_DIFFERENCES:
        difference_scales = mechanics_scales(list(MECHANICS_DIFFERENCES.values()), symmetric = True)
        for label, dataset in MECHANICS_DIFFERENCES.items():
            target, reference = MAP_PAIRS[label]
            plotter = CICEPlotter(**contexts(target))
            path = EXPORT / (f"{label}_{region}_mechanics_difference_{START}_{END}.png")
            saved = plot_mechanics_triptych(dataset, plotter, f"{target} - {reference}", path, difference_scales, difference = True)
            register(saved, "shuga PyGMT mechanics difference triptych", [target, reference], (START, END), region)
            print(f"Saved: {saved}")

## 3. Actual lateral-drag stresses and activation

I'll verify the archived executable and history definitions before treating KuxE/KuyE and KuxN/KuyN as actual face stresses. Accepted stress units are converted to Pa. Estimated rho_i*hi*Ku stresses are not used here.

I'll use shuga's component-magnitude helper separately at each collocated E or N face, followed by its temporal-mean calculation. The Q90 reduction remains a small notebook-specific operation. E and N fields remain separate: no vector is assembled from components at different faces.

These diagnostics use the archived daily-history components. Taking their magnitude before averaging across days does not recover the mean subdaily stress magnitude if the components were already averaged during history output.

For each face, I'll export the mean, Q90, maximum, valid-sample count and fraction of valid samples above a declared stress threshold. Valid zero stresses remain included; missing component pairs remain missing. The above-threshold fraction describes the archived stress signal, not necessarily the internal activation state of the parameterisation.

Geographic maps require reviewed longitude/latitude fields for each face. I'll specify their variable names in STRESS_FACE_COORDS. If these are unavailable, the numerical reduction can retain the original T-cell-index crop, explicitly labelled as such, but geographic plotting remains disabled. T-cell coordinates will not be substituted for face locations.

I'll use shuga's PyGMT triptych generator for mean, Q90 and above-threshold fraction maps. Stress panels share a scale across experiments and faces; fraction panels use 0–1.

These are face-specific diagnostics, not a closed momentum budget or FI-conditioned stresses. Comparing them quantitatively with T-cell FI, atmospheric/ocean stresses or closure regimes requires separate checks of locations, sampling and mapping. Phase-space analysis is not performed in this section.

In [13]:
import shuga.metrics.diagnostics as mdiag
import shuga.metrics.calculations as mcalc

# Stress exceedance is a diagnostic threshold, not a closure activation switch.
STRESS_ACTIVE_THRESHOLD_PA = 0.0

# Each entry is (longitude_variable, latitude_variable) in NATIVE_AREA_PATH.
# Supply reviewed E/N face coordinates; do not substitute TLON/TLAT.
STRESS_FACE_COORDS = {"E": None, "N": None}
RUN_STRESS_MAPS    = False  # Enable only after supplying reviewed face coordinates.
STRESS_DATA = {}
STRESS_ROWS = []

if RUN_STRESS:
    if not LD_STRESS_DEFINITION_VERIFIED:
        raise ValueError("Confirm executable provenance and actual stress definitions")
    if NATIVE_AREA_PATH is None or not NATIVE_GRID_VERIFIED:
        raise ValueError("Confirm the native grid and face locations")
    if (not np.isfinite(STRESS_ACTIVE_THRESHOLD_PA) or STRESS_ACTIVE_THRESHOLD_PA < 0):
        raise ValueError("Stress threshold must be finite and non-negative")
    required = {"TLON", "TLAT"}
    for coordinates in STRESS_FACE_COORDS.values():
        if coordinates is not None:
            if len(coordinates) != 2:
                raise ValueError("Face coordinates must be a (longitude, latitude) pair")
            required.update(coordinates)
    with open_product(NATIVE_AREA_PATH) as source:
        missing = required - set(source.variables)
        if missing:
            raise KeyError(f"Native-grid file lacks: {sorted(missing)}")
        native = source[sorted(required)].load()
    region = DIAGNOSTIC_REGION
    bounds = ANTARCTIC_8_REGIONS[region]["plot_region"]
    face_components = {"E": ("KuxE", "KuyE"),
                       "N": ("KuxN", "KuyN")}
    # Only actual stress units are accepted.
    stress_factors = {"pa": 1.0,
                      "n/m2": 1.0,
                      "n/m^2": 1.0,
                      "nm-2": 1.0,
                      "nm^-2": 1.0,
                      "kpa": 1000.0}
    implementation = {module.__name__: wf.file_sha256(module.__file__) for module in (mdiag, mcalc)}
    for name in DIAGNOSTIC_CASES:
        history = load(name, "history", ["KuxE", "KuyE", "KuxN", "KuyN"])
        wf.daily_check(history, START, END)
        for face, components in face_components.items():
            coordinate_names = STRESS_FACE_COORDS[face]
            if coordinate_names is None:
                lon, lat = native.TLON, native.TLAT
                crop_basis = "T-cell-index crop; face coordinates unavailable"
            else:
                lon = native[coordinate_names[0]]
                lat = native[coordinate_names[1]]
                crop_basis = "geographic crop using reviewed face coordinates"
            tile = wf.crop_region(history[list(components)], lon, lat, bounds)
            # Standardise both components before using shuga's magnitude helper.
            normalised = {}
            for variable in components:
                field = tile[variable]
                unit = (str(field.attrs.get("units", "")).lower().replace(" ", "").replace("²", "2"))
                if unit not in stress_factors:
                    raise ValueError(f"{name}/{variable}: verify actual stress units {field.attrs.get('units')!r}")
                converted = field * stress_factors[unit]
                converted.attrs = dict(field.attrs, units="Pa")
                normalised[variable] = converted
            face_history = xr.Dataset(normalised)
            valid = (np.isfinite(face_history[components[0]]) & np.isfinite(face_history[components[1]]))
            face_history = face_history.where(valid)
            magnitude = mdiag.mag(face_history, *components, name="stress_magnitude")
            magnitude.attrs.update(units = "Pa", long_name = f"{face}-face archived stress magnitude")
            mean = mcalc.compute_temporal_mean(magnitude, name = "mean", long_name = "Mean archived stress magnitude")
            quantile_input = magnitude
            if magnitude.chunks is not None:
                quantile_input = magnitude.chunk({"time": -1})
            q90 = (quantile_input.quantile(0.90, dim = "time", skipna = True).drop_vars("quantile", errors="ignore"))
            maximum = magnitude.max("time", skipna = True)
            # Missing component pairs are excluded from the denominator.
            exceedance = xr.where(np.isfinite(magnitude), (magnitude > STRESS_ACTIVE_THRESHOLD_PA).astype("float32"), np.nan)
            fraction = mcalc.compute_persistence_mask(exceedance, name = "above_threshold_fraction",
                                                      long_name = "Fraction of valid records above stress threshold",
                                                      percent = False)

            out = xr.Dataset({"mean"                         : mean,
                              "q90"                          : q90,
                              "maximum"                      : maximum,
                              "above_threshold_fraction"     : fraction,
                              "valid_sample_count"           : magnitude.notnull().sum("time"),
                              "above_threshold_sample_count" : (magnitude > STRESS_ACTIVE_THRESHOLD_PA).sum("time")})
            for variable in ("mean", "q90", "maximum"):
                out[variable].attrs["units"] = "Pa"
            for variable in ("valid_sample_count", "above_threshold_sample_count"):
                out[variable].attrs["units"] = "1"
            if coordinate_names is not None:
                coordinates = wf.crop_region(
                    xr.Dataset({"lon": lon, "lat": lat}), lon, lat, bounds)
                out, coordinates = xr.align(out, coordinates, join = "exact")
                out = out.assign_coords(lon = coordinates.lon, lat = coordinates.lat)
            out.attrs.update(experiment     = name,
                             face           = face,
                             source_components = ", ".join(components),
                             crop_basis     = crop_basis,
                             start          = START,
                             end            = END,
                             threshold_pa   = float(STRESS_ACTIVE_THRESHOLD_PA),
                             interpretation = "Magnitude of archived daily-history components")
            out = out.compute()
            valid_count = int(out.valid_sample_count.sum().item())
            active_count = int(out.above_threshold_sample_count.sum().item())
            STRESS_ROWS.append({"experiment": name,
                                "face": face,
                                "region": region,
                                "crop_basis": crop_basis,
                                "maximum_daily_history_magnitude_pa": float(out.maximum.max(skipna=True).item()),
                                "valid_face_samples": valid_count,
                                "above_threshold_face_samples": active_count,
                                "pooled_sample_exceedance_fraction": (active_count / valid_count if valid_count else np.nan),
                                "threshold_pa": STRESS_ACTIVE_THRESHOLD_PA})
            path = EXPORT / (f"{name}_{region}_{face}_face_stress_{START}_{END}.nc")
            signature = wf.signature({"analysis": SIGNATURE,
                                      "stage": "face-stress-v2",
                                      "implementation": implementation,
                                      "experiment": name,
                                      "face": face,
                                      "region": region,
                                      "coordinates": coordinate_names,
                                      "threshold_pa": STRESS_ACTIVE_THRESHOLD_PA,
                                      "history_attrs": dict(history.attrs)})
            wf.save_cached(out, path, signature)
            STRESS_DATA[(name, face, region)] = out
            register(path, "actual face-stress temporal summaries", [name], (START, END), f"{face} face; {crop_basis}")
    summary = pd.DataFrame(STRESS_ROWS)
    export_table(summary, "face_stress_summary", DIAGNOSTIC_CASES, (START, END))
    display(summary)
else:
    print("Stress diagnostics disabled: RUN_STRESS=False")

Stress diagnostics disabled: RUN_STRESS=False


In [14]:
from shuga.plotting.cice import CICEPlotter

if RUN_STRESS and RUN_STRESS_MAPS and STRESS_DATA:
    for key, dataset in STRESS_DATA.items():
        if "lon" not in dataset.coords or "lat" not in dataset.coords:
            raise ValueError(f"{key}: supply reviewed STRESS_FACE_COORDS and rerun the computation cell before geographic plotting")
    # One common stress scale for all experiments, faces and stress panels.
    maxima = []
    for dataset in STRESS_DATA.values():
        for variable in ("mean", "q90"):
            values = dataset[variable].values
            finite = values[np.isfinite(values)]
            if finite.size:
                maxima.append(float(finite.max()))
    if not maxima:
        raise ValueError("No finite stress values to plot")
    upper = max(maxima) or 1.0
    stress_series = [0.0, upper, upper / 20]
    fraction_series = [0.0, 1.0, 0.05]
    for (name, face, region), dataset in STRESS_DATA.items():
        plotter = CICEPlotter(**contexts(name))
        bounds = ANTARCTIC_8_REGIONS[region]["plot_region"]
        specifications = [("mean", "Mean stress magnitude", "cmocean/turbid", stress_series, "Stress magnitude (Pa)"),
                          ("q90", "Q90 stress magnitude", "cmocean/turbid", stress_series, "Stress magnitude (Pa)"),
                          ("above_threshold_fraction", "Above-threshold fraction", "viridis", fraction_series,f"Fraction above {STRESS_ACTIVE_THRESHOLD_PA:g} Pa")]
        panels = []
        for variable, title, cmap, series, colourbar in specifications:
            panels.append({"title": f"{name}, {face} face: {title}",
                           "layers": [{"data": dataset[variable],
                                       "lon": dataset.lon,
                                       "lat": dataset.lat,
                                       "mask_zero": False,
                                       "cmap": cmap,
                                       "series": series,
                                       "colorbar": {"position": "JBC+w6c/0.3c+h+o0c/0.8c",
                                                    "frame": [f"xaf+l{colourbar}"]}}]})
        path = EXPORT / (f"{name}_{region}_{face}_face_stress_{START}_{END}.png")
        saved = plotter.plot_triptych(region      = bounds,
                                      panels      = panels,
                                      fig_size    = 10.0,
                                      panel_gap   = "2.5c",
                                      output_path = path)
        register(saved, "shuga PyGMT actual face-stress triptych", [name], (START, END), f"{region}; {face} face")
        print(f"Saved: {saved}")
elif RUN_STRESS and STRESS_DATA:
    print("Numerical stress products are available. Geographic plotting requires reviewed face coordinates and RUN_STRESS_MAPS=True.")

## 4. Formation, retention, retreat and tendencies

I'll use shuga's persistence calculation for annual FI fractions and its area-rate integration for thermodynamic and dynamic tendencies. The first/last classified-day reductions remain in the notebook because they describe the selected classification record rather than an existing shuga event-date metric.

For each calendar year, I'll export valid-day counts, FI-day counts, FI fraction, and the first/last qualifying day of year. Day-of-year values are one-based and retain leap days. A valid cell with no FI has zero occurrence and missing first/last dates; a cell with no valid classification samples has missing occurrence and persistence. Partial years are identified explicitly.

These quantities do not establish independently observed formation or retreat dates. First and last occurrence can bracket fragmented occupancy, and FI-day counts do not measure uninterrupted retention. I'll inspect the daily sequence and retain the observation-composite definitions when making event comparisons.

For tendencies, I'll use the same integration as shuga's FITAR and FIMAR metrics. Its helper expects area-fraction tendency per day, so I'll convert known input units explicitly before integration. Actual native cell areas are required.

I'll retain classification-coverage information alongside each tendency. If a tendency is missing at any selected FI cell, that day's integrated value remains missing rather than becoming a partial sum. Unknown classification cells are excluded and their area reported; the result therefore applies to the known classified support.

The exported tendencies are in 10³ km²/day. They are contributions evaluated inside the contemporaneous FI mask, not the derivative of total classified FIA. Mask transitions and any unresolved sampling effects prevent these sums from constituting a closed FIA budget.

Optional annual maps use shuga's PyGMT triptych generator. Seasonal FIA growth/retreat rates remain separate from these first/last occurrence diagnostics.

In [16]:
import shuga.metrics.calculations as mcalc
from shuga.metrics.regional import ensure_2d_static

OCCURRENCE_PRODUCTS = {}
TENDENCY_DATA = {}

# ------------------------------------------------------------------
# Annual classification occurrence and persistence
# ------------------------------------------------------------------
if RUN_OCCURRENCE:
    for name in DIAGNOSTIC_CASES:
        mask = wf.binary_mask(load(name, "mask", ["FI_mask"]).FI_mask)
        wf.daily_check(mask, START, END)
        for year, part in mask.groupby("time.year"):
            year = int(year)
            valid_days = part.notnull().sum("time")
            fi_days = (part == 1).sum("time").where(valid_days > 0)
            fi_fraction = mcalc.compute_persistence_mask(part, name = "FI_fraction",
                                                         long_name = "Annual FI fraction over valid classification days",
                                                         percent = False)
            # Actual calendar day-of-year, including leap days.
            doy = part.time.dt.dayofyear.astype("float32")
            first = doy.where(part == 1).min("time", skipna=True)
            last = doy.where(part == 1).max("time", skipna=True)
            dates = pd.DatetimeIndex(part.time.values)
            calendar_days = len(pd.date_range(f"{year}-01-01", f"{year}-12-31"))
            complete_year = (dates[0].normalize() == pd.Timestamp(year, 1, 1)
                             and dates[-1].normalize() == pd.Timestamp(year, 12, 31)
                             and len(dates) == calendar_days)
            out = xr.Dataset({"valid_days"  : valid_days,
                              "FI_days"     : fi_days,
                              "FI_fraction" : fi_fraction,
                              "first_doy"   : first,
                              "last_doy"    : last})
            for variable in ("valid_days", "FI_days"):
                out[variable].attrs["units"] = "days"
            for variable in ("first_doy", "last_doy"):
                out[variable].attrs.update(units = "1", description = "One-based calendar day of year")
            actual_start = str(dates[0].date())
            actual_end   = str(dates[-1].date())
            out.attrs.update(experiment             = name,
                             classification         = METHOD,
                             year                   = year,
                             start                  = actual_start,
                             end                    = actual_end,
                             calendar_year_complete = int(complete_year),
                             calendar_days          = calendar_days,
                             requested_days         = len(dates),
                             interpretation         = ("Classification occurrence; not independently validated"
                                                       "formation, retention or retreat dates"))
            path = EXPORT / (f"{name}_occurrence_{year}_{actual_start}_{actual_end}.nc")
            signature = wf.signature({"analysis": SIGNATURE,
                                      "stage": "annual-occurrence-v2",
                                      "experiment": name,
                                      "year": year,
                                      "period": [actual_start, actual_end],
                                      "method": METHOD,
                                      "shuga_calculations": wf.file_sha256(mcalc.__file__)})
            wf.save_cached(out.compute(), path, signature)
            OCCURRENCE_PRODUCTS[(name, year)] = path
            register(path, "annual classification occurrence and persistence", [name], (actual_start, actual_end),
                     f"Complete calendar year: {complete_year}")
            print(f"Saved: {path}")

# ------------------------------------------------------------------
# FI-conditioned area tendencies: shuga FITAR/FIMAR integration
# ------------------------------------------------------------------
if RUN_TENDENCIES:
    if (NATIVE_AREA_PATH is None or not NATIVE_GRID_VERIFIED or not MASK_TEMPORAL_SUPPORT_VERIFIED):
        raise ValueError("Review native areas/grid and tendency-mask temporal support")
    with open_product(NATIVE_AREA_PATH) as native:
        area = ensure_2d_static(native.tarea).load()
    area            = wf.area_m2(area)
    spatial_dims    = list(area.dims)
    total_grid_area = area.sum(spatial_dims)
    # Convert known units to concentration fraction per day.
    daily_factors = {"s-1": 86400.0, "s^-1": 86400.0, "s**-1": 86400.0, "1/s": 86400.0,
                     "1/day": 1.0, "day-1": 1.0, "day^-1": 1.0, "d-1": 1.0, "d^-1": 1.0,
                     "%/day": 0.01, "%day-1": 0.01, "%day^-1": 0.01, "%d-1": 0.01, "%d^-1": 0.01,
                     "%/s": 864.0}
    definitions = {"FITAR": ("daidtt", "FI-conditioned thermodynamic area tendency"),
                   "FIMAR": ("daidtd", "FI-conditioned dynamic area tendency")}
    for name in DIAGNOSTIC_CASES:
        history = load(name, "history", ["daidtt", "daidtd"])
        mask = wf.binary_mask(load(name, "mask", ["FI_mask"]).FI_mask)
        history, mask = wf.align_daily(history, mask, start = START, end = END)
        history, mask, cell_area = xr.align(history, mask, area, join = "exact")
        expected_dims = {"time", *spatial_dims}
        if set(mask.dims) != expected_dims:
            raise ValueError(f"{name}: classification dimensions do not match cell areas")
        fi = mask == 1
        known_area = cell_area.where(mask.notnull(), 0.0).sum(spatial_dims)
        selected_fi_area = cell_area.where(fi, 0.0).sum(spatial_dims)
        result = {"classified_grid_area": known_area / 1e9,
                  "unclassified_grid_area": (total_grid_area - known_area) / 1e9,
                  "selected_FI_cell_area": selected_fi_area / 1e9}
        for metric, (variable, long_name) in definitions.items():
            field = history[variable]
            if set(field.dims) != expected_dims:
                raise ValueError(f"{name}/{variable}: unexpected dimensions {field.dims}")
            unit = (str(field.attrs.get("units", "")).lower().replace(" ", ""))
            if unit not in daily_factors:
                raise ValueError(f"{name}/{variable}: review tendency units {field.attrs.get('units')!r}")
            daily = field * daily_factors[unit]
            daily.attrs = dict(field.attrs, units="day^-1")
            finite = np.isfinite(daily)
            missing_fi_cells = (fi & ~finite).sum(spatial_dims)
            # Existing shuga integration, after exact alignment and conversion.
            integrated = mcalc.compute_area_rate(daily.where(finite), cell_area, fi, name = metric, long_name = long_name)
            # Do not publish a partial sum as a complete FI tendency.
            integrated = integrated.where((missing_fi_cells == 0) & (known_area > 0)) / 1e9
            integrated.attrs.update(units = "10^3 km^2 day^-1", source_variable = variable,
                                    source_units = str(field.attrs.get("units")),
                                    interpretation = "Known classified support; not a closed FIA budget")
            result[metric] = integrated
            result[f"{metric}_missing_FI_cells"] = missing_fi_cells
        dataset = xr.Dataset(result).compute()
        TENDENCY_DATA[name] = dataset
        table = dataset.to_dataframe().reset_index()
        table["area_rate_units"] = "10^3 km^2 day^-1"
        table["area_units"] = "10^3 km^2"
        table["interpretation"] = ("FI-conditioned tendencies on known classification support; not a closed FIA budget")
        export_table(table, "FI_tendencies_" + name, [name], (START, END))
        display(table.head())

In [17]:
from shuga.plotting.cice import CICEPlotter

RUN_OCCURRENCE_MAPS = False  # Enable after generating the annual products.

if RUN_OCCURRENCE_MAPS:
    if NATIVE_AREA_PATH is None or not NATIVE_GRID_VERIFIED:
        raise ValueError("Review native T-grid coordinates before plotting")
    if not OCCURRENCE_PRODUCTS:
        raise ValueError("Run the occurrence computation first")
    with open_product(NATIVE_AREA_PATH) as source:
        grid = source[["TLON", "TLAT"]].load()
    region = DIAGNOSTIC_REGION
    bounds = ANTARCTIC_8_REGIONS[region]["plot_region"]
    tile_grid = wf.crop_region(grid, grid.TLON, grid.TLAT, bounds)
    specifications = [("FI_days", "FI-day count", [0, 366, 30], "Days classified FI"),
                      ("first_doy", "First qualifying day", [1, 366, 30], "Day of year"),
                      ("last_doy", "Last qualifying day", [1, 366, 30], "Day of year")]
    for (name, year), product in OCCURRENCE_PRODUCTS.items():
        with open_product(product) as source:
            data = wf.crop_region(source, grid.TLON, grid.TLAT, bounds).load()
        plotter = CICEPlotter(**contexts(name))
        partial = ("" if data.attrs["calendar_year_complete"] else " (partial year)")
        panels = []
        for variable, title, series, colourbar in specifications:
            panels.append({"title": f"{name}, {year}{partial}: {title}",
                           "layers": [{"data": data[variable],
                                       "lon": tile_grid.TLON,
                                       "lat": tile_grid.TLAT,
                                       "mask_zero": False,
                                       "cmap": "viridis",
                                       "series": series,
                                       "colorbar": {"position": "JBC+w6c/0.3c+h+o0c/0.8c",
                                                    "frame": [f"xaf+l{colourbar}"]} }] })
        path = EXPORT / (f"{name}_{region}_occurrence_{year}_{data.attrs['start']}_{data.attrs['end']}.png")
        saved = plotter.plot_triptych(region = bounds, panels = panels, fig_size = 10.0, panel_gap = "2.5c", output_path = path)
        register(saved, "shuga PyGMT annual occurrence triptych", [name], (data.attrs["start"], data.attrs["end"]), region)
        print(f"Saved: {saved}")

## 5. Geometry and distance from anchors

I'll use the executed form-factor file selected by FORM_FACTOR_PATH, without regenerating geometry. For the primary comparison, I'll use the combined Cs-high geometry as a common reference for coastline-only, iceberg-only and combined experiments, together with the matched controls and tensile cases. Changing the reference geometry between experiments would change the distance bins as well as the model response.

Shuga provides the persistence and area-weighted reduction methods. I'll retain the notebook's nearest-anchor distance helper because I did not find an equivalent diagnostic in shuga.

An anchor-bearing cell is a source T cell where either FFx or FFy is positive. Distance is the great-circle distance to the nearest such cell centre. It is not distance to the actual coastline, iceberg perimeter, mapped velocity-face stress, or an ocean-only route. Distances are calculated from the full reference geometry before selecting the analysis region, so nearby anchors outside the region remain available.

I'll verify the form-factor coordinates against the native T grid and supply an explicit binary ocean mask. Statistics use the selected region's geo_region bounds, rather than its plotting bounds. Land cells are excluded.

For each experiment, I'll calculate fractional persistence using shuga's existing method and summarise it within the same distance bins using actual cell-area weights. The output includes bin area and cell count, so small or empty bins are visible. Zero persistence remains valid. This comparison requires complete daily classification support in the selected ocean domain; missing classifications will not silently alter the denominator.

The geometry comparison includes cst-drag, GIB-drag and Cs-high. Distance for a drag-disabled control still refers to the reference geometry, not to active drag in that experiment.

I'll interpret the curves as descriptive spatial comparisons. Bins finer than the model spacing do not add spatial resolution, neighbouring cells are not independent samples, and distance alone does not establish a causal anchoring response. Any figure will use PyGMT.

In [ ]:
import shuga.metrics.calculations as mcalc
from shuga.metrics.regional import region_mask
from shuga.grid.geometry import normalise_longitudes, area_to_m2

# Explicit analysis choices.
ANCHOR_REGION              = DIAGNOSTIC_REGION
ANCHOR_OCEAN_MASK_VARIABLE = "ocean_mask"  # Binary: 1=ocean, 0=land.
ANCHOR_FF_LON_VARIABLE     = "lon"         # Coordinates in the executed FF file.
ANCHOR_FF_LAT_VARIABLE     = "lat"
ANCHOR_DISTANCE_EDGES_KM   = [0, 10, 25, 50, 100, np.inf]

# Include the geometry experiments absent from DIAGNOSTIC_CASES.
ANCHOR_CASES = list(dict.fromkeys(list(DIAGNOSTIC_CASES) + ["cst-drag", "GIB-drag", "Cs-high"]))
ANCHOR_TABLE = None

def anchor_degrees(field, longitude=False):
    """Explicit coordinate-unit conversion; never infer units from magnitude."""
    unit = str(field.attrs.get("units", "")).lower()
    if "radian" in unit:
        out = np.rad2deg(field)
    elif "degree" in unit:
        out = field.copy(deep=False)
    else:
        raise ValueError(f"{field.name}: verify coordinate units {field.attrs.get('units')!r}")
    if longitude:
        out = normalise_longitudes(out, to="-180-180")
    out.attrs = dict(field.attrs, units="degrees")
    return out

if RUN_ANCHOR:
    if (not FORM_FACTOR_GRID_VERIFIED or not NATIVE_GRID_VERIFIED or FORM_FACTOR_PATH is None or NATIVE_AREA_PATH is None):
        raise ValueError("Review the reference form-factor file, native grid and ocean mask")
    # FORM_FACTOR_PATH must be the common reference geometry for all cases.
    with open_product(NATIVE_AREA_PATH) as source:
        required = ["TLON", "TLAT", "tarea", ANCHOR_OCEAN_MASK_VARIABLE]
        missing = set(required) - set(source.variables)
        if missing:
            raise KeyError(f"Native-grid file lacks {sorted(missing)}. Supply a reviewed binary ocean mask.")
        native = source[required].load()
    with xr.open_dataset(FORM_FACTOR_PATH) as source:
        required = ["FFx", "FFy", ANCHOR_FF_LON_VARIABLE, ANCHOR_FF_LAT_VARIABLE]
        missing = set(required) - set(source.variables)
        if missing:
            raise KeyError(f"Form-factor file lacks {sorted(missing)}; configure its actual coordinate variable names.")
        ff = source[required].load()
    lon = anchor_degrees(native.TLON, longitude=True)
    lat = anchor_degrees(native.TLAT)
    ff_lon = anchor_degrees(ff[ANCHOR_FF_LON_VARIABLE], longitude = True)
    ff_lat = anchor_degrees(ff[ANCHOR_FF_LAT_VARIABLE])
    if lon.ndim != 2:
        raise ValueError("Expected a two-dimensional native T grid")
    for field in (lat, native.tarea, native[ANCHOR_OCEAN_MASK_VARIABLE], ff.FFx, ff.FFy, ff_lon, ff_lat):
        if set(field.dims) != set(lon.dims):
            raise ValueError(f"{field.name}: dimensions do not match the native T grid")
    # Align by named dimensions and exact indexes, not shape alone.
    ff = ff.transpose(*lon.dims)
    ff_lon = ff_lon.transpose(*lon.dims)
    ff_lat = ff_lat.transpose(*lon.dims)
    native, ff = xr.align(native, ff, join="exact")
    longitude_error = normalise_longitudes(ff_lon - lon, to = "-180-180")
    if not (np.allclose(longitude_error, 0, atol=1e-4, rtol=0) and np.allclose(ff_lat, lat, atol=1e-4, rtol=0)):
        raise ValueError("Form-factor coordinates do not match the native T grid")
    for variable in ("FFx", "FFy"):
        values = ff[variable].values
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError(f"{variable}: reference factors must be finite and non-negative")
    # Require known area units before calling shuga's converter.
    area_units = (str(native.tarea.attrs.get("units", "")).lower().replace(" ", "").replace("²", "2"))
    if area_units not in {"m2", "m^2", "cm2", "cm^2", "km2", "km^2"}:
        raise ValueError("Verify native cell-area units")
    area = wf.area_m2(area_to_m2(native.tarea, target="tarea"))
    ocean = wf.binary_mask(native[ANCHOR_OCEAN_MASK_VARIABLE])
    selected_region = region_mask(lon, lon, lat, {ANCHOR_REGION: ANTARCTIC_8_REGIONS[ANCHOR_REGION]}).sel(region=ANCHOR_REGION, drop=True)
    if bool((selected_region & ocean.isnull()).any().item()):
        raise ValueError("Ocean mask has missing values within the region")
    domain = selected_region & (ocean == 1)
    if not bool(domain.any().item()):
        raise ValueError("Selected region contains no valid ocean cells")
    # The distance helper is retained; no new geometry is generated.
    anchors = ((ff.FFx > 0) | (ff.FFy > 0)).astype(float)
    if not bool(anchors.any().item()):
        raise ValueError("Reference geometry contains no anchor-bearing cells")
    required_coordinates = domain | (anchors == 1)
    if bool((required_coordinates & (~np.isfinite(lon) | ~np.isfinite(lat))).any().item()):
        raise ValueError("Missing coordinates at query or anchor cells")
    # Calculate against the full source geometry before regional selection.
    distance = wf.anchor_distance_km(lon, lat, anchors)
    grid = xr.Dataset({"distance": distance, "area": area, "domain": domain}).where(selected_region, drop=True)
    domain = grid.domain == 1
    edges = np.asarray(ANCHOR_DISTANCE_EDGES_KM, dtype=float)
    if (len(edges) < 2 or edges[0] != 0 or not np.all(np.diff(edges) > 0) or not np.isposinf(edges[-1])):
        raise ValueError("Distance edges must increase from 0 to +infinity")
    geometry_sha256 = wf.file_sha256(FORM_FACTOR_PATH)
    rows = []
    for name in ANCHOR_CASES:
        if name not in EXPERIMENTS:
            raise ValueError(f"Out-of-scope experiment: {name}")
        mask = wf.binary_mask(load(name, "mask", ["FI_mask"]).FI_mask)
        wf.daily_check(mask, START, END)
        # Verify the full spatial grid before selecting the regional subset.
        xr.align(mask.isel(time=0, drop=True), area, join="exact")
        mask = mask.sel({dim: grid[dim] for dim in area.dims})
        mask, domain = xr.align(mask, domain, join="exact")
        if bool((domain & mask.isnull().any("time")).any().compute().item()):
            raise ValueError(f"{name}: incomplete daily classification in the ocean domain; "
                             "review coverage before comparing distance-bin means")
        fip = mcalc.compute_persistence_mask(mask.where(domain), name = "FIP",
                                             long_name = "FI fraction over the selected model period", percent = False).compute()

        for bin_index, (lower, upper) in enumerate(zip(edges[:-1], edges[1:])):
            inside = (domain & (grid.distance >= lower) & (grid.distance < upper))
            # This shuga helper is a generic area-weighted field reduction.
            # It preserves the input units and reports valid weighted area.
            summary = mcalc.compute_area_weighted_stress(fip, grid.area, inside, base_name = "FIP").compute()
            label = (f"{lower:g}-{upper:g}" if np.isfinite(upper) else f"{lower:g}+")
            rows.append({"experiment": name,
                         "region": ANCHOR_REGION,
                         "bin_index": bin_index,
                         "distance_bin_km": label,
                         "lower_km": lower,
                         "upper_km": upper,
                         "mean_fip": float(summary.FIP_mean.item()),
                         "valid_area_km2": float(summary.FIP_valid_area_m2.item()) / 1e6,
                         "ocean_cell_count": int(inside.sum().item()),
                         "start": START,
                         "end": END,
                         "reference_geometry": str(FORM_FACTOR_PATH),
                         "reference_geometry_sha256": geometry_sha256})
    ANCHOR_TABLE = pd.DataFrame(rows)
    export_table(ANCHOR_TABLE, "anchor_distance_" + ANCHOR_REGION, ANCHOR_CASES, (START, END))
    display(ANCHOR_TABLE)
else:
    print("Anchor-distance analysis disabled: RUN_ANCHOR=False")

In [ ]:
import tempfile
from pathlib import Path
import pygmt

RUN_ANCHOR_PLOT = False

# Main geometry comparison; add other analysed cases if useful.
ANCHOR_PLOT_CASES = ["cst-drag", "GIB-drag", "Cs-high"]

if RUN_ANCHOR_PLOT:
    if ANCHOR_TABLE is None:
        raise ValueError("Run the anchor-distance calculation first")
    pens = ["1.5p,blue", "1.5p,orange", "1.5p,black"]
    bins = (ANCHOR_TABLE[["bin_index", "distance_bin_km"]].drop_duplicates().sort_values("bin_index"))
    with tempfile.TemporaryDirectory() as temporary:
        ticks = Path(temporary) / "distance_bins.txt"
        ticks.write_text("".join(f"{row.bin_index} a {row.distance_bin_km}\n" for row in bins.itertuples()))
        fig = pygmt.Figure()
        fig.basemap(region     = [-0.5, len(bins) - 0.5, 0, 1],
                    projection = "X14c/8c",
                    frame      = [f"pxc{ticks}+lDistance to reference anchor-cell centre (km)",
                                  "pya0.2f0.1+lArea-weighted FI fraction",
                                  f"WSen+t{ANCHOR_REGION}: {START} to {END}"])
        for index, name in enumerate(ANCHOR_PLOT_CASES):
            table = ANCHOR_TABLE[ANCHOR_TABLE.experiment == name].sort_values("bin_index")
            if table.empty:
                raise ValueError(f"No anchor-distance results for {name}")
            # Preserve gaps at empty bins rather than joining across them.
            for _, segment in table.groupby(table.mean_fip.isna().cumsum()):
                segment = segment.dropna(subset=["mean_fip"])
                if segment.empty:
                    continue
                fig.plot(x = segment.bin_index, y = segment.mean_fip, pen = pens[index % len(pens)])
            valid = table.dropna(subset=["mean_fip"])
            if not valid.empty:
                fig.plot(x = valid.bin_index, y = valid.mean_fip, style = "c0.16c",
                         fill = pens[index % len(pens)].split(",")[1], pen = pens[index % len(pens)], label = name)
        fig.legend(position = "JTR+jTR+o0.2c", box = "+gwhite+p0.5p")
        path = EXPORT / (f"{ANCHOR_REGION}_anchor_distance_{START}_{END}.png")
        fig.savefig(path)
    register(path, "PyGMT persistence by reference-anchor distance", ANCHOR_PLOT_CASES, (START, END), ANCHOR_REGION)
    print(f"Saved: {path}")

## 6. Pack and total sea-ice checks

I'll use shuga's existing area, volume, thickness and diagnostic reductions, together with its stored PI_mask classification. I'll reconstruct Tc speed on the full periodic grid before spatial reductions.

The fixed geographical domain must be chosen independently of the simulated outcome. Within that same domain, I'll compare all ice-covered cells against each experiment's changing pack subset. Neither selection has constant ice coverage, so I'll retain the sampled areas alongside the means.

Thickness uses shuga's concentration-and-cell-area weighting. Speed and deformation use cell-area weighting over valid selected samples. I'll retain signed divergence separately from the non-negative deformation proxy; cancellation in mean divergence is not evidence of weak deformation.

I'll interpret hi according to the reviewed history definition before calling shuga's volume and thickness methods. Converting grid-cell volume per area to ice-covered-area thickness avoids applying concentration twice.

Total quantities refer to the Southern Hemisphere ocean domain, not just the fixed pack-check region. SIA follows shuga's concentration threshold convention. I'll also export SIA_all_concentrations to distinguish the original unthresholded area integral. SIV and SIT retain shuga's all-concentration integration convention.

Missing concentration or ice thickness within the total ocean domain must be resolved before reporting totals. Diagnostic means retain their valid sampled areas. Missing FI/PI classifications in the fixed domain will not silently become pack ice.

I'll use shuga's existing PyGMT SIA and SIT climatology plotters. These figures describe model contrasts and interannual variability. Observational skill requires separately supplied, appropriately sampled observations over the common evaluation window; it is not established by the model-only figures below.

In [ ]:
import shuga.metrics.calculations as mcalc
import shuga.metrics.diagnostics as mdiag
import shuga.regridding.cice as cgrid
from shuga.grid.geometry import area_to_m2
from shuga.metrics.regional import ensure_2d_static

PACK_OCEAN_MASK_VARIABLE   = "ocean_mask"  # Binary: 1=ocean, 0=land.
FIXED_PACK_DOMAIN_VERIFIED = False      # Review physical alignment and independent domain selection.
PACK_RESULTS               = {}

def pack_convert(field, factors, units):
    key = str(field.attrs.get("units", "")).lower().replace(" ", "")
    if key not in factors:
        raise ValueError(f"{field.name}: verify units {field.attrs.get('units')!r}")
    result = field * factors[key]
    result.attrs = dict(field.attrs, units=units)
    return result

if RUN_PACK:
    if (FIXED_PACK_MASK_PATH is None or NATIVE_AREA_PATH is None or HI_CONVENTION is None or not NATIVE_GRID_VERIFIED
        or not FIXED_PACK_DOMAIN_VERIFIED):
        raise ValueError("Review the fixed geographical domain, native ocean grid and hi convention first")
    with open_product(NATIVE_AREA_PATH) as source:
        native = source[["tarea", "TLAT", PACK_OCEAN_MASK_VARIABLE]].load()
    area_raw = ensure_2d_static(native.tarea)
    area_unit = (str(area_raw.attrs.get("units", "")).lower().replace(" ", "").replace("²", "2"))
    if area_unit not in {"m2", "m^2", "cm2", "cm^2", "km2", "km^2"}:
        raise ValueError("Verify native cell-area units")
    area = wf.area_m2(area_to_m2(area_raw, target="tarea"))
    ocean = wf.binary_mask(ensure_2d_static(native[PACK_OCEAN_MASK_VARIABLE]))
    latitude = ensure_2d_static(native.TLAT)
    latitude_unit = str(latitude.attrs.get("units", "")).lower()
    if "radian" in latitude_unit:
        latitude = np.rad2deg(latitude)
    elif "degree" not in latitude_unit:
        raise ValueError("Verify native latitude units")
    if bool(((latitude < 0) & ocean.isnull()).any().item()):
        raise ValueError("Ocean mask has missing values in the SH domain")
    total_domain = (latitude < 0) & (ocean == 1)
    with open_product(FIXED_PACK_MASK_PATH) as source:
        fixed = wf.binary_mask(ensure_2d_static(source.fixed_pack_mask)).load()
    fixed, area, total_domain = xr.align(fixed, area, total_domain, join = "exact")
    if bool((fixed.isnull() & total_domain).any().item()):
        raise ValueError("Fixed-domain mask is missing over SH ocean cells")
    if bool(((fixed == 1) & ~total_domain).any().item()):
        raise ValueError("Fixed pack domain includes land or non-SH cells")
    fixed_domain = fixed == 1
    if not bool(fixed_domain.any().item()):
        raise ValueError("Fixed pack domain is empty")
    spatial_dims = list(area.dims)
    threshold = float(CFG["aice_thresh"])
    velocity_names = ["uvelE", "uvelN", "vvelE", "vvelN"]
    velocity_factors = {"m/s": 1.0, "ms-1": 1.0, "ms^-1": 1.0, "cm/s": 0.01, "cms-1": 0.01, "cms^-1": 0.01}
    strain_factors = {"s-1": 1.0, "s^-1": 1.0, "s**-1": 1.0, "1/s": 1.0,
                      "1/day": 1 / 86400, "day-1": 1 / 86400, "d-1": 1 / 86400,
                      "%/day": 0.01 / 86400, "%day-1": 0.01 / 86400, "%d-1": 0.01 / 86400}
    for name in DIAGNOSTIC_CASES:
        h = load(name, "history", ["hi", "aice", "divu", "shear"] + velocity_names)
        fi = wf.binary_mask(load(name, "mask", ["FI_mask"]).FI_mask)
        pi = wf.binary_mask(load(name, "mask", ["PI_mask"], ice_type = "PI").PI_mask)
        h, fi, pi = wf.align_daily(h, fi, pi, start = START, end = END)
        h, fi, pi, cell_area = xr.align(h, fi, pi, area, join = "exact")
        # Standardise concentration before using the shared reductions.
        concentration = pack_convert(h.aice, {"1": 1.0, "fraction": 1.0, "unitless": 1.0, "dimensionless": 1.0,
                                              "%": 0.01, "percent": 0.01}, "1")
        h = h.assign(aice=concentration)
        hi = wf.ice_thickness(h, HI_CONVENTION)
        checks = xr.Dataset({"invalid_concentration": (total_domain & (~np.isfinite(concentration) | (concentration < 0) | (concentration > 1))).any(),
                             "invalid_thickness": (total_domain & (concentration > 0) & (~np.isfinite(hi) | (hi < 0))).any(),
                             "missing_classification": (fixed_domain & (fi.isnull() | pi.isnull())).any(),
                             "inconsistent_pack_mask": (fixed_domain & fi.notnull() & pi.notnull() & ((pi == 1) != ((concentration > threshold) & (fi == 0)))).any()}).compute()
        failed = [key for key, value in checks.data_vars.items() if bool(value.item())]
        if failed:
            raise ValueError(f"{name}: review {', '.join(failed)}")
        fixed_ice = fixed_domain & (concentration > threshold)
        fixed_pack = fixed_domain & (pi == 1)
        # Existing shuga reconstruction, evaluated before any spatial crop.
        velocity = xr.Dataset({variable: pack_convert(h[variable], velocity_factors, "m/s") for variable in velocity_names})
        speed = cgrid.compute_tgrid_speed(velocity, h.aice, grid_type = "Tc", wrap_x = True, cgrid_combine = "mean")
        # Exclude missing contributing velocity components from the means.
        indicators = [np.isfinite(h[variable]).astype("float32") for variable in velocity_names]
        valid_e, valid_n = cgrid.c2t(*indicators, h.aice, wrap_x=True, combine="mean", nan_to_zero=True)
        speed = speed.where((valid_e == 1) & (valid_n == 1))
        speed.attrs["units"] = "m/s"
        deformation = xr.Dataset({variable: pack_convert(h[variable], strain_factors, "s-1") for variable in ("divu", "shear")})
        strain = mdiag.strain_invariant(deformation)
        strain.attrs["units"] = "s-1"
        result = {}
        for label, selection in (("fixed", fixed_ice), ("pack", fixed_pack)):
            # Match shuga's PIT/SIT concentration-and-area weighting.
            result[f"{label}_hi"] = mcalc.compute_thickness_series(concentration, hi, cell_area, selection,
                                                                   name      = f"{label}_hi",
                                                                   long_name = f"{label}: concentration/area-weighted thickness")
            result[f"{label}_ice_cell_area"] = (cell_area.where(selection, 0).sum(spatial_dims) / 1e9)
            result[f"{label}_ice_cell_area"].attrs["units"] = "10^3 km^2"
            fields = {"speed": speed,
                      "divu": deformation.divu,
                      "shear": deformation.shear,
                      "strain_proxy": strain}
            for variable, field in fields.items():
                # Generic area-weighted mean and its valid sampled area.
                field = field.where(np.isfinite(field))
                reduced = mcalc.compute_area_weighted_stress(field, cell_area, selection, base_name = f"{label}_{variable}")
                result[f"{label}_{variable}"] = reduced[f"{label}_{variable}_mean"]
                result[f"{label}_{variable}_valid_area_m2"] = reduced[f"{label}_{variable}_valid_area_m2"]
        # Standard shuga SIA uses its >= threshold SI presence convention.
        result["SIA"] = mcalc.compute_area_series(concentration, cell_area, total_domain & (concentration >= threshold),
                                                  name = "SIA", long_name = "SH sea-ice area at the configured SI threshold",
                                                  scale = 1e9)
        # Retain the old unthresholded definition under a distinct name.
        result["SIA_all_concentrations"] = mcalc.compute_area_series(concentration, cell_area, total_domain,
                                                                     name      = "SIA_all_concentrations",
                                                                     long_name = "SH ocean concentration integral without an ice threshold",
                                                                     scale     = 1e9)
        result["SIV"] = mcalc.compute_volume_series(concentration, hi, cell_area, total_domain,
                                                    name = "SIV", long_name = "SH sea-ice volume", scale = 1e12)
        result["SIT"] = mcalc.compute_thickness_series(concentration, hi, cell_area, total_domain,
                                                       name = "SIT", long_name = "SH concentration/area-weighted sea-ice thickness")
        dataset = xr.Dataset(result).compute()
        dataset.attrs.update(experiment     = name,
                             total_domain   = "Southern Hemisphere ocean",
                             pack_domain    = "PI_mask intersected with the fixed geographical domain",
                             fixed_mask     = str(FIXED_PACK_MASK_PATH),
                             hi_convention  = HI_CONVENTION,
                             aice_threshold = threshold)
        PACK_RESULTS[name] = dataset
        # NetCDF preserves variable units and definitions alongside the CSV.
        path = EXPORT / f"pack_{name}_{START}_{END}.nc"
        wf.save_cached(dataset, path, wf.signature({"analysis"         : SIGNATURE,
                                                    "stage"            : "shuga-pack-checks-v2",
                                                    "experiment"       : name,
                                                    "hi_convention"    : HI_CONVENTION,
                                                    "fixed_mask"       : str(FIXED_PACK_MASK_PATH),
                                                    "fixed_mask_mtime" : Path(FIXED_PACK_MASK_PATH).stat().st_mtime_ns,
                                                    "calculations"     : wf.file_sha256(mcalc.__file__),
                                                    "regridding"       : wf.file_sha256(cgrid.__file__)}))
        register(path, "shuga pack and SH total sea-ice checks", [name], (START, END))
        export_table(dataset.to_dataframe().reset_index(), "pack_" + name, [name], (START, END))
        print(f"Saved: {path}")
else:
    print("Pack checks disabled: RUN_PACK=False")

In [ ]:
from shuga.plotting.cice import CICEPlotter

RUN_PACK_PLOTS  = False
PACK_PLOT_CASES = list(PRIMARY)

if RUN_PACK_PLOTS:
    missing = set(PACK_PLOT_CASES) - set(PACK_RESULTS)
    if missing:
        raise ValueError(f"Run pack reductions first for {sorted(missing)}")

    plotter = CICEPlotter(**contexts(PACK_PLOT_CASES[0]))
    palette = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]
    colors = {name: palette[index % len(palette)] for index, name in enumerate(PACK_PLOT_CASES)}
    # Shuga's SIA plotter labels 10^6 km²; exports above use 10^3 km².
    sia = pd.concat({name: PACK_RESULTS[name].SIA.to_series() / 1000.0 for name in PACK_PLOT_CASES}, axis = 1)
    path = EXPORT / f"SH_SIA_climatology_{START}_{END}.png"
    plotter.plot_sia_daily_climatology_envelope(sia, path,
                                                start_date  = START,
                                                end_date    = END,
                                                envelope    = "minmax",
                                                smooth_days = 1,
                                                order       = PACK_PLOT_CASES,
                                                colors      = colors,
                                                y_min       = 0,
                                                y_max       = max(1.0, float(np.ceil(np.nanmax(sia.values) * 1.1))),
                                                title       = f"Model SH SIA: {START} to {END}",
                                                write_csv   = True)
    register(path, "shuga PyGMT model SIA climatology", PACK_PLOT_CASES, (START, END))
    titles = {"SIT": "SH total ice thickness",
              "fixed_hi": "Ice thickness within the fixed geographical domain",
              "pack_hi": "Pack thickness within the fixed geographical domain"}
    for variable, title in titles.items():
        series = {name: PACK_RESULTS[name][variable].to_series() for name in PACK_PLOT_CASES}
        finite_parts = [s.to_numpy()[np.isfinite(s.to_numpy())] for s in series.values()]
        finite_parts = [values for values in finite_parts if values.size]
        if not finite_parts:
            print(f"No finite values for {variable}; skipping figure")
            continue
        upper = max(1.0, float(np.ceil(np.concatenate(finite_parts).max() * 1.1)))
        path = EXPORT / (f"{variable}_monthly_climatology_{START}_{END}.png")
        plotter.plot_sit_monthly_climatology_envelope(series, path,
                                                      envelope = "minmax",
                                                      order    = PACK_PLOT_CASES,
                                                      colors   = colors,
                                                      y_min    = 0,
                                                      y_max    = upper,
                                                      title    = f"{title}: {START} to {END}",
                                                      write_csv = True)
        register(path, "shuga PyGMT model thickness climatology", PACK_PLOT_CASES, (START, END), variable)
        print(f"Saved: {path}")

## 7. Point thickness comparison

I'll use shuga's processed BEPSII point product where available, retaining its measured ice thickness, campaign identifiers, observation keys and QA fields. Core length is not a substitute for ice thickness. Snow depth remains supporting metadata; it is not added to the ice-thickness comparison.

I'll first inspect the number of usable fast-ice observations within the model period and their distribution across campaigns and dates. Records outside that period remain in the audit but do not enter contemporaneous validation. No nearest-date or cross-era substitution is made.

The nearest ocean cell is selected once, independently of experiment and simulated ice conditions. I'll report its coordinates and great-circle offset, and compare results under declared distance limits. This does not establish that the observation and model cell share an unobstructed ocean connection or resolve the same coastal feature.

I'll retain model concentration and FI classification at every eligible matchup. Thickness scores use the same observations across experiments, restricted to samples where all selected experiments have valid ice-present thickness. These are conditional thickness scores. I'll report ice presence and FI recovery separately over all spatially eligible observations, so missed ice is not hidden by that restriction. Model FI classification is not itself a requirement for entering the thickness comparison.

Repeated cores from the same campaign, model cell and day are averaged before scoring. I'll use shuga's existing skill statistics within campaigns, then calculate campaign-balanced bias, MAE and RMSE. These remain descriptive sample statistics: core locations are not a random or area-representative sample of Antarctic fast ice.

I'll retain thick observations and diagnostic QA flags rather than removing inconvenient values. Duplicate identifiers and inconsistent metadata require review. Nearby-cell thickness ranges provide representativeness context, not confidence intervals and not a way to select the best-fitting model cell.

Point-comparison figures will use PyGMT. I will not claim robust validation from a large core count if the evidence comes from only a few campaigns or model cell-days.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

POINT_INPUT               = (Path(POINT_OBS_PATH).expanduser() if POINT_OBS_PATH is not None else 
                             Path("/g/data/gv90/da1339/SeaIce/InSitu/processed/BEPSII_TS/SH/fast/BEPSII_TS_fast_points.zarr"))
POINT_CASES               = list(DIAGNOSTIC_CASES)
POINT_REFERENCE           = "free-slip-no-drag"
POINT_DISTANCE_LIMITS_KM  = [10.0, 25.0, 50.0]
POINT_PRIMARY_DISTANCE_KM = 25.0  # Declared matching choice; inspect sensitivity.
POINT_MIN_AICE            = float(CFG["aice_thresh"])
POINT_NEIGHBOURS          = 4
POINT_OCEAN_MASK_VARIABLE = "ocean_mask"  # Or a reviewed binary tmask.
POINT_INPUT_AUDIT         = pd.DataFrame()
POINT_MATCHUPS            = pd.DataFrame()
POINT_SCORES              = pd.DataFrame()
POINT_GROUPED             = {}

if RUN_POINT_OBS:
    if POINT_INPUT.suffix.lower() == ".csv":
        points = pd.read_csv(POINT_INPUT)
        definition = "User-reviewed ice-only thickness in metres"
    else:
        with open_product(POINT_INPUT) as source:
            required = {"time", "latitude", "longitude", "ice_thickness", "observation_key", "cruise", "is_fast_ice"}
            missing = required - set(source.variables)
            if missing:
                raise KeyError(f"Expected shuga processed point schema; missing {sorted(missing)}")
            if source.ice_thickness.attrs.get("units") != "m":
                raise ValueError("Review observational thickness units")
            points      = source.to_dataframe().reset_index()
            definition  = source.attrs.get("thickness_definition", "")
            source_name = source.attrs.get("source", str(POINT_INPUT))
        points = points.rename(columns={"latitude": "lat",
                                        "longitude": "lon",
                                        "ice_thickness": "thickness_m",
                                        "snow_thickness": "snow_m",
                                        "observation_key": "observation_id",
                                        "cruise": "campaign"})
        points["ice_type"] = np.where(points.is_fast_ice == 1, "fast", "other")
        points["source"] = source_name
    required = {"observation_id", "campaign", "time", "lon", "lat", "thickness_m", "ice_type", "source"}
    missing = required - set(points.columns)
    if missing:
        raise ValueError(f"Point input lacks {sorted(missing)}")
    points = points.reset_index(drop=True)
    for column in ("observation_id", "campaign", "source", "ice_type"):
        points[column] = (points[column].astype("string").fillna("").str.strip())
    points["ice_type"] = points.ice_type.str.lower().replace({"fast_ice": "fast", "fi": "fast"})
    for column in ("lon", "lat", "thickness_m"):
        points[column] = pd.to_numeric(points[column], errors="coerce")
    dates = pd.to_datetime(points.time, errors="coerce", format="mixed")
    if (not pd.api.types.is_datetime64_any_dtype(dates.dtype) or isinstance(dates.dtype, pd.DatetimeTZDtype)):
        raise ValueError("Supply reviewed sampling calendar dates without ambiguous time zones")
    points["day"]                  = dates.dt.normalize()
    points["input_path"]           = str(POINT_INPUT)
    points["thickness_definition"] = definition
    points["exclusion_reason"]     = ""
    def point_reject(condition, reason):
        condition = pd.Series(condition, index=points.index).fillna(True)
        points.loc[condition, "exclusion_reason"] += reason + ";"
    point_reject(points.day.isna(), "invalid_date")
    point_reject(~points.day.between(pd.Timestamp(START), pd.Timestamp(END)), "outside_model_period")
    point_reject(~np.isfinite(points.lon) | ~np.isfinite(points.lat) | ~points.lon.between(-180, 360)
                 | ~points.lat.between(-90, 0, inclusive="left"), "invalid_SH_location")
    point_reject(~np.isfinite(points.thickness_m) | (points.thickness_m <= 0), "invalid_ice_thickness")
    point_reject(points.ice_type != "fast", "not_observed_fast_ice")
    point_reject(points.campaign == "", "missing_campaign")
    point_reject(points.source == "", "missing_source")
    point_reject(points.observation_id == "", "missing_observation_id")
    point_reject(points.observation_id.duplicated(keep=False), "duplicate_identifier_requires_review")
    # Existing BEPSII duplicate/partial-core/thick-ice flags are retained.
    # A processor's retained duplicate flag is not itself grounds for deletion.
    POINT_INPUT_AUDIT = points
    export_table(points, "point_observation_inventory", [], (START, END))
    eligible = points[points.exclusion_reason == ""]
    inventory = eligible.groupby("campaign").agg(n_observations = ("observation_id", "size"),
                                                 first_date     = ("day", "min"),
                                                 last_date      = ("day", "max"))
    display(inventory)
    print(f"Eligible observations: {len(eligible)} / {len(points)}")
    if eligible.empty:
        print("No eligible contemporaneous observations. Do not substitute observations from other years.")

In [ ]:
from scipy.spatial import cKDTree

if RUN_POINT_OBS and not POINT_INPUT_AUDIT.empty:
    if (not POINT_OBS_SAMPLING_VERIFIED or not NATIVE_GRID_VERIFIED or NATIVE_AREA_PATH is None or HI_CONVENTION is None):
        raise ValueError("Review observation definitions/sampling, native grid and hi convention")
    if (POINT_PRIMARY_DISTANCE_KM not in POINT_DISTANCE_LIMITS_KM or min(POINT_DISTANCE_LIMITS_KM) <= 0):
        raise ValueError("Review the primary and sensitivity distance limits")
    pool = POINT_INPUT_AUDIT[POINT_INPUT_AUDIT.exclusion_reason == ""].copy()
    if not pool.empty:
        with open_product(NATIVE_AREA_PATH) as source:
            native = source[["TLON", "TLAT", POINT_OCEAN_MASK_VARIABLE]].load()
        def point_degrees(field):
            units = str(field.attrs.get("units", "")).lower()
            if "radian" in units:
                return np.rad2deg(field)
            if "degree" in units:
                return field
            raise ValueError(f"{field.name}: verify coordinate units")
        lon = point_degrees(native.TLON)
        lat = point_degrees(native.TLAT)
        ocean = wf.binary_mask(native[POINT_OCEAN_MASK_VARIABLE])
        lon, lat, ocean = xr.align(lon, lat, ocean, join="exact")
        def point_xyz(longitude, latitude):
            lo, la = np.deg2rad(longitude), np.deg2rad(latitude)
            return np.stack([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)], axis=-1)
        xyz = point_xyz(lon.values, lat.values)
        good = (np.isfinite(xyz).all(-1) & (ocean.values == 1) & (lat.values < 0))
        flat = np.flatnonzero(good)
        if not len(flat):
            raise ValueError("No valid SH ocean cells in the native grid")
        tree = cKDTree(xyz[good])
        k = min(POINT_NEIGHBOURS, len(flat))
        chord, position = tree.query(point_xyz(pool.lon.to_numpy(), pool.lat.to_numpy()), k = k)
        chord = np.asarray(chord).reshape(len(pool), k)
        position = np.asarray(position).reshape(len(pool), k)
        distance = 2 * 6371.0 * np.arcsin(np.clip(chord / 2, 0, 1))
        j, i = np.unravel_index(flat[position], lon.shape)
        POINT_INPUT_AUDIT.loc[pool.index, "grid_distance_km"] = distance[:, 0]
        too_far = distance[:, 0] > max(POINT_DISTANCE_LIMITS_KM)
        POINT_INPUT_AUDIT.loc[pool.index[too_far], "exclusion_reason"] += "beyond_maximum_matching_distance;"
        keep = ~too_far
        pool = pool.loc[keep].copy()
        distance, j, i = distance[keep], j[keep], i[keep]
        export_table(POINT_INPUT_AUDIT, "point_spatial_matching_audit", [], (START, END))
        frames = []
        if not pool.empty:
            coordinates  = {"sample": pool.observation_id.to_numpy(), "neighbor": np.arange(k)}
            sample_times = xr.DataArray(pool.day.to_numpy(), dims="sample", coords={"sample": coordinates["sample"]})
            indexers     = {lon.dims[0]: xr.DataArray(j, dims=("sample", "neighbor"), coords=coordinates),
                            lon.dims[1]: xr.DataArray(i, dims=("sample", "neighbor"), coords=coordinates)}
            nearby       = xr.DataArray(distance <= POINT_PRIMARY_DISTANCE_KM, dims=("sample", "neighbor"), coords=coordinates)
            for name in POINT_CASES:
                h           = load(name, "history", ["hi", "aice"])
                fi          = load(name, "mask", ["FI_mask"]).FI_mask
                h, fi       = wf.align_daily(h, fi, start=START, end=END)
                xr.align(h.hi.isel(time=0, drop=True), lon, join="exact")
                selected    = h.sel(time=sample_times).isel(indexers)
                selected_fi = wf.binary_mask(fi.sel(time=sample_times).isel(indexers))
                unit        = str(selected.aice.attrs.get("units", "")).lower().replace(" ", "")
                factors     = {"1": 1.0, "fraction": 1.0, "unitless": 1.0, "dimensionless": 1.0, "%": 0.01, "percent": 0.01}
                if unit not in factors:
                    raise ValueError(f"{name}: review concentration units")
                aice = selected.aice * factors[unit]
                aice.attrs["units"] = "1"
                selected = selected.assign(aice=aice)
                sampled = xr.Dataset({"model_hi"   : wf.ice_thickness(selected, HI_CONVENTION),
                                      "model_aice" : aice,
                                      "model_FI"   : selected_fi}).compute()

                if bool(((sampled.model_aice < 0) | (sampled.model_aice > 1)).any().item()):
                    raise ValueError(f"{name}: concentration outside 0–1")
                neighbour_hi = sampled.model_hi.where(nearby & (sampled.model_aice > POINT_MIN_AICE)
                                                      & np.isfinite(sampled.model_hi) & (sampled.model_hi >= 0))
                frame = pool.copy()
                frame["experiment"] = name
                frame["model_j"] = j[:, 0]
                frame["model_i"] = i[:, 0]
                frame["grid_distance_km"] = distance[:, 0]
                frame["model_lon"] = lon.values[j[:, 0], i[:, 0]]
                frame["model_lat"] = lat.values[j[:, 0], i[:, 0]]
                frame["model_thickness_m"] = sampled.model_hi.isel(neighbor=0).values
                frame["model_aice"] = sampled.model_aice.isel(neighbor=0).values
                frame["model_FI"] = sampled.model_FI.isel(neighbor=0).values
                frame["model_ice_present"] = (frame.model_aice > POINT_MIN_AICE)
                frame["score_thickness_m"] = frame.model_thickness_m.where(frame.model_ice_present
                                                                           & np.isfinite(frame.model_thickness_m)
                                                                           & (frame.model_thickness_m >= 0))
                frame["nearby_min_thickness_m"] = neighbour_hi.min("neighbor", skipna=True).values
                frame["nearby_max_thickness_m"] = neighbour_hi.max("neighbor", skipna=True).values
                frame["nearby_valid_ice_cells"] = neighbour_hi.notnull().sum("neighbor").values
                frame["ice_presence_threshold"] = POINT_MIN_AICE
                frames.append(frame)
        if frames:
            POINT_MATCHUPS = pd.concat(frames, ignore_index=True)
            export_table(POINT_MATCHUPS, "point_thickness_matchups", POINT_CASES, (START, END))
            display(POINT_MATCHUPS.head())
        else:
            print("No observations remain within the maximum matching distance.")

In [ ]:
from shuga.metrics.skill import skill_stats

if not POINT_MATCHUPS.empty:
    blocks  = ["campaign", "day", "model_j", "model_i"]
    support = POINT_MATCHUPS.pivot(index   = "observation_id",
                                   columns = "experiment",
                                   values  = "score_thickness_m").reindex(columns=POINT_CASES)
    common_ids = support.index[support.notna().all(axis=1)]
    POINT_MATCHUPS["common_thickness_support"] = (POINT_MATCHUPS.observation_id.isin(common_ids))
    score_rows = []
    for radius in POINT_DISTANCE_LIMITS_KM:
        available = POINT_MATCHUPS[POINT_MATCHUPS.grid_distance_km <= radius]
        common = available[available.common_thickness_support]
        grouped = common.groupby(["experiment"] + blocks, as_index = False).agg(observed_m     = ("thickness_m", "mean"),
                                                                                model_m        = ("score_thickness_m", "mean"),
                                                                                model_FI       = ("model_FI", "mean"),
                                                                                n_observations = ("observation_id", "size"))
        POINT_GROUPED[radius] = grouped
        for name in POINT_CASES:
            all_samples = available[available.experiment == name]
            matched = grouped[grouped.experiment == name]
            # Recovery diagnostics use all spatially eligible observations.
            recovery = all_samples.groupby(blocks).agg(ice_present  = ("model_ice_present", "mean"),
                                                       FI_recovered = ("model_FI", "mean"))
            campaign_recovery = recovery.groupby("campaign").mean()
            campaign_errors = []
            for campaign, part in matched.groupby("campaign"):
                with np.errstate(divide="ignore", invalid="ignore"):
                    stats = skill_stats(part.model_m.to_numpy(), part.observed_m.to_numpy())
                campaign_errors.append(stats)
            errors = pd.DataFrame(campaign_errors)
            has_errors = not errors.empty
            score_rows.append({"experiment": name,
                               "distance_limit_km"                      : radius,
                               "n_spatially_eligible_observations"      : len(all_samples),
                               "n_common_thickness_observations"        : (int(matched.n_observations.sum()) if len(matched) else 0),
                               "n_common_cell_days"                     : len(matched),
                               "n_common_campaigns"                     : matched.campaign.nunique(),
                               "ice_presence_fraction_campaign_balanced": (campaign_recovery.ice_present.mean()),
                               "FI_recovery_fraction_campaign_balanced" : (campaign_recovery.FI_recovered.mean()),
                               "FI_classification_known_fraction"       : (all_samples.model_FI.notna().mean()),
                               "Bias_campaign_balanced_m"               : (errors.Bias.mean() if has_errors else np.nan),
                               "MAE_campaign_balanced_m"                : (errors.MAE.mean() if has_errors else np.nan),
                               # Equal campaign weight applied to squared error.
                               "RMSE_campaign_balanced_m": (np.sqrt(np.mean(errors.RMSE**2)) if has_errors else np.nan) })
    POINT_SCORES = pd.DataFrame(score_rows)
    POINT_SCORES["delta_MAE_vs_reference_m"] = np.nan
    for radius in POINT_DISTANCE_LIMITS_KM:
        reference = POINT_SCORES[(POINT_SCORES.experiment == POINT_REFERENCE) & (POINT_SCORES.distance_limit_km == radius)]
        if not reference.empty:
            baseline = reference.MAE_campaign_balanced_m.iloc[0]
            select = POINT_SCORES.distance_limit_km == radius
            POINT_SCORES.loc[select, "delta_MAE_vs_reference_m"] = (POINT_SCORES.loc[select, "MAE_campaign_balanced_m"] - baseline)
    export_table(POINT_SCORES, "point_thickness_campaign_balanced_scores", POINT_CASES, (START, END))
    export_table(POINT_MATCHUPS, "point_thickness_matchups_with_support", POINT_CASES, (START, END))
    display(POINT_SCORES)
    if not len(common_ids):
        print("No common ice-present thickness support across the selected experiments. Report recovery diagnostics; no thickness ranking.")


In [ ]:
import pygmt

RUN_POINT_OBS_PLOTS = False

if RUN_POINT_OBS_PLOTS:
    grouped = POINT_GROUPED.get(POINT_PRIMARY_DISTANCE_KM, pd.DataFrame())
    if grouped.empty:
        raise ValueError("No common-support grouped matchups to plot")
    upper = max(1.0, float(np.ceil(grouped[["observed_m", "model_m"]].to_numpy().max() * 1.1)))
    for name in POINT_CASES:
        data = grouped[grouped.experiment == name]
        if data.empty:
            continue
        fig = pygmt.Figure()
        fig.basemap(region     = [0, upper, 0, upper],
                    projection = "X11c/11c",
                    frame      = [f"WSen+t{name}: campaign-cell-day means", "xaf+lObserved ice thickness (m)", "yaf+lModel ice-covered-area thickness (m)"])
        fig.plot(x = [0, upper], y = [0, upper], pen = "1p,gray50,--")
        selections = [(data.model_FI == 1, "darkgreen", "Model classified FI"), (data.model_FI == 0, "orange", "Model not classified FI"),
                      (data.model_FI.isna(), "gray50", "FI classification missing")]
        for selection, color, label in selections:
            part = data[selection]
            if not part.empty:
                fig.plot(x = part.observed_m, y = part.model_m, style = "c0.18c", fill = color, pen = "0.3p,black", label = label)
        fig.legend(position = "JTL+jTL+o0.2c", box = "+gwhite+p0.5p")
        path = EXPORT / (f"{name}_point_thickness_{POINT_PRIMARY_DISTANCE_KM:g}km_{START}_{END}.png")
        fig.savefig(path)
        register(path, "PyGMT contemporaneous point thickness comparison", [name], (START, END), "Common ice-present support; campaign-cell-day means")
        print(f"Saved: {path}")
        

## 8. Export record and close sources

I'll use these exports to complete the spatial/mechanistic discussion and replace the development figures, with shared scales and clear conditioning. Publication map styling can reuse these verified numerical products. Missing observations or diagnostics remain limitations; a successful inventory run is not scientific validation. Return to the main notebook's review record after inspecting the outputs.

In [ ]:
(EXPORT/'supporting_artifacts.json').write_text(json.dumps(ARTIFACTS, indent = 2))
(EXPORT/'supporting_review.json').write_text(json.dumps({'FIP_enabled'                : RUN_FIP,
                                                         'FIP_support_reviewed'       : FIP_GRID_AND_SUPPORT_VERIFIED,
                                                         'mechanics_enabled'          : RUN_MECHANICS,
                                                         'hi_convention'              : HI_CONVENTION,
                                                         'actual_stress_enabled'      : RUN_STRESS,
                                                         'stress_definition_reviewed' : LD_STRESS_DEFINITION_VERIFIED,
                                                         'pack_enabled'               : RUN_PACK,
                                                         'point_obs_enabled'          : RUN_POINT_OBS,
                                                         'geometry_enabled'           : RUN_ANCHOR,
                                                         'tendencies_enabled'         : RUN_TENDENCIES,
                                                         'signature'                  : SIGNATURE},
                                                        indent=2))
for ds in OPEN.values():
    ds.close()
OPEN.clear()
print('Sources closed; disabled stages remain pending.')